# AttackAware PolyIoM v1.1.4: every results figure, generated from code

This notebook recomputes the study's results from the stored embeddings and
the sealed keys, and then draws every results figure from what it has just
computed. No number is typed in, and nothing is read from an earlier result
file except to compare against it.

| Stage | Computes | Figures |
|---|---|---|
| 1 | the 80-setting development sweep, and the selection rule re-run on it | 3, 4, S2 |
| 2 | the protected and unprotected score histograms | A, B, C, D |
| 3 | identity-cluster bootstrap intervals; protection cost against the unprotected system | 5, P |
| 4 | the ablation of the polynomial | 6 |
| 5 | the full-knowledge inversion attack | 8 |
| 6 | the revocation test over 40 fresh keys | 7 |
| 7 | a comparison of every regenerated number with the study's original output | |
| 8 | all twelve figures, as PNG and PDF, and a zip to download | |

**What it needs on Drive** (under `MyDrive/AttackAware_PolyIoM_v1_1_4/`, as the
study left it): `embeddings/`, `protocol/`, `seal/`, and from `runs/` the
selected keys, the frozen IoM tensors, the sealed held-out and external
results and the two sealed sweeps.

**What it writes.** Everything it computes goes to `runs_regenerated/`, and the
figures to `figures_regenerated/`. It never writes to `runs/` or `seal/`.

**Checks.** Each stage stops if it cannot reproduce the sealed result it
depends on: the selection rule must choose the sealed setting, the rebuilt
scores must give the sealed held-out and external point estimates, and the
ablation must reproduce the sealed held-out EER. Stage 7 then compares every
regenerated number with the study's original output where that is on Drive.

**Run time.** Stages 5 and 6 run the inversion attack on the CPU, as the study
did, and take most of the time. Every stage saves its result, so after a
disconnect, run all cells again and finished stages are skipped.

**Use:** Runtime, then Run all.


In [ ]:
#@title 1. Settings, Drive and input check
import hashlib, importlib.util, json, math, os, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

try:
    from google.colab import drive, files
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:                      # outside Colab, e.g. a local check
    IN_COLAB = False

PROJECT = Path(os.environ.get(
    "POLYIOM_PROJECT", "/content/drive/MyDrive/AttackAware_PolyIoM_v1_1_4"))
RUNS_IN = PROJECT / "runs"               # the study's outputs: read only
WORK = PROJECT / "runs_regenerated"      # everything this notebook computes
FIGURES_OUT = PROJECT / "figures_regenerated"
RUNTIME = Path(os.environ.get("POLYIOM_RUNTIME", "/content/polyiom_runtime"))

DIR = {"protocol": PROJECT / "protocol", "embeddings": PROJECT / "embeddings",
       "runs": WORK, "seal": PROJECT / "seal"}
S0, G = 2026, 5
DEVICE = torch.device("cpu")
N_BOOTSTRAP, CONFIDENCE = 2000, 0.95

LFW_FACE_TRIALS = DIR["protocol"] / "lfw_face_trials.tsv"
LFW_EMB = DIR["embeddings"] / "lfw_all_valid_embeddings.npz"
LIBRI_MANIFEST = DIR["protocol"] / "librispeech_internal.tsv"
LIBRI_EMB = DIR["embeddings"] / "librispeech_internal_embeddings.npz"
HELDOUT_RESULTS = {m: RUNS_IN / "heldout" / m / "heldout_result.json"
                   for m in ("face", "voice")}
ABLATION_OUTPUT = WORK / "ablation" / "ablation_result.json"

DIMENSION = {"face": 512, "voice": 192}
REQUIRED = [LFW_FACE_TRIALS, LFW_EMB, LIBRI_MANIFEST, LIBRI_EMB,
            DIR["seal"] / "operating_point_face.json",
            DIR["seal"] / "operating_point_voice.json"]
for m, d in DIMENSION.items():
    REQUIRED += [RUNS_IN / "key_search" / m / "selected_key.json",
                 HELDOUT_RESULTS[m]]
    REQUIRED += [RUNS_IN / "iom_tensors" /
                 f"{m}_o{o}_k{1 + math.ceil((d - G) / (G - o))}.pt"
                 for o in range(5)]
missing = [str(p) for p in REQUIRED if not p.is_file()]
if missing:
    raise FileNotFoundError("These inputs are missing:\n- " + "\n- ".join(missing))

OPTIONAL = {
    "external VCTK embeddings": DIR["embeddings"] / "vctk_external_embeddings.npz",
    "sealed external result": RUNS_IN / "external/voice/external_result.json",
    "sealed sweeps (to compare)": RUNS_IN / "sweep/voice/sweep_80.csv",
}
for label, path in OPTIONAL.items():
    print(f"  {label:<28} {'found' if path.is_file() else 'not found - skipped'}")

torch.use_deterministic_algorithms(True, warn_only=True)
WORK.mkdir(parents=True, exist_ok=True)
RUNTIME.mkdir(parents=True, exist_ok=True)
print(f"\nInputs from {RUNS_IN}\nOutputs to  {WORK}\nFigures to  {FIGURES_OUT}")


In [ ]:
%%writefile {RUNTIME}/ablation_only.py
# 2. The study's pipeline primitives and the ablation
"""Self-contained ablation runtime for AttackAware PolyIoM v1.1.4.

Loaded by AttackAware_PolyIoM_v1_1_4_ABLATION.ipynb after its preflight cell
has defined PROJECT, DIR, S0, G, DEVICE, the embedding paths, HELDOUT_RESULTS
and ABLATION_OUTPUT.

Rebuilds the frozen score histograms from the stored embeddings and keys, so
it depends on no other notebook. Writes one file, runs/ablation/
ablation_result.json, and no seal.
"""

def sha256_file(path, chunk=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while block := handle.read(chunk):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(obj, indent=2, sort_keys=True, default=str))
    os.replace(temporary, path)


def H(*fields):
    payload = "|".join(str(field) for field in fields).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "big") & ((1 << 63) - 1)


def seed_unlink(modality, key_index):
    return H(S0, "UNLINK", modality, key_index)


def l2_np(vector, eps=1e-12):
    vector = np.asarray(vector, dtype=np.float32)
    norm = np.linalg.norm(vector)
    if norm <= eps:
        raise RuntimeError("Zero-norm embedding")
    return (vector / norm).astype(np.float32)


def selected_poly_key(modality):
    path = DIR["runs"] / "key_search" / modality / "selected_key.json"
    obj = json.loads(path.read_text())
    return np.asarray(obj["C"], np.float32), np.asarray(obj["E"], np.int64)


def face_heldout_data():
    trials = pd.read_csv(LFW_FACE_TRIALS, sep="\t")
    stored = np.load(LFW_EMB, allow_pickle=True)
    embeddings = stored["embedding"].astype(np.float32)
    held = trials[trials.partition == "evaluation"]
    subjects, enroll, probes = [], {}, {}
    for uid, group in held.groupby("subject_id"):
        erow = group[group.role == "enroll"]
        prows = group[group.role == "probe"]
        if len(erow) != 1 or len(prows) < 1:
            continue
        uid = str(uid)
        subjects.append(uid)
        enroll[uid] = embeddings[int(erow.iloc[0].embedding_index)]
        probes[uid] = [
            embeddings[int(row.embedding_index)] for row in prows.itertuples()
        ]
    return sorted(subjects), enroll, probes


def voice_heldout_data():
    manifest = pd.read_csv(LIBRI_MANIFEST, sep="\t", dtype={"speaker_id": str})
    stored = np.load(LIBRI_EMB, allow_pickle=True)
    embeddings = stored["embedding"].astype(np.float32)
    table = pd.DataFrame({
        "speaker_id": stored["speaker_id"].astype(str),
        "rank": stored["rank"].astype(int),
        "embedding_index": np.arange(len(embeddings), dtype=np.int64),
    })
    held = manifest.merge(
        table[["speaker_id", "rank", "embedding_index"]],
        on=["speaker_id", "rank"], how="inner",
    )
    held = held[held.partition == "evaluation"]
    subjects, enroll, probes = [], {}, {}
    for uid, group in held.groupby("speaker_id"):
        group = group.sort_values("rank")
        if len(group) != 15:
            continue
        uid = str(uid)
        members = np.stack([
            embeddings[int(row.embedding_index)]
            for row in group[group["rank"] < 5].itertuples()
        ])
        subjects.append(uid)
        enroll[uid] = l2_np(members.mean(axis=0))
        probes[uid] = [
            embeddings[int(row.embedding_index)]
            for row in group[group["rank"] >= 5].itertuples()
        ]
    return sorted(subjects), enroll, probes


def poly_indices(d, overlap, device):
    stride = G - overlap
    k = 1 + math.ceil((d - G) / stride)
    idx = torch.arange(k, device=device)[:, None] * stride + torch.arange(G, device=device)
    mask = idx < d
    return idx.clamp(max=d - 1).long(), mask


def poly_transform(vectors, coefficients, exponents, overlap):
    idx, mask = poly_indices(vectors.shape[-1], overlap, vectors.device)
    values = vectors[..., idx] * mask.to(vectors.dtype)
    coefficients = torch.as_tensor(coefficients, device=vectors.device, dtype=vectors.dtype)
    exponents = torch.as_tensor(exponents, device=vectors.device, dtype=torch.int64)
    return (torch.pow(values, exponents) * coefficients).sum(dim=-1)


def load_iom_tensor(modality, overlap, d):
    k = 1 + math.ceil((d - G) / (G - overlap))
    path = DIR["runs"] / "iom_tensors" / f"{modality}_o{overlap}_k{k}.pt"
    if not path.is_file():
        raise FileNotFoundError(f"Frozen IoM tensor is missing: {path}")
    try:
        tensor = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        tensor = torch.load(path, map_location="cpu")
    if (tensor.ndim != 3 or tensor.shape[0] < 256 or tensor.shape[1] < 32
            or tensor.shape[-1] != k):
        raise RuntimeError(f"Frozen IoM tensor has an invalid shape: {path}")
    return tensor


print("Frozen-pipeline primitives ready.")


ARMS = ("polyiom", "iom_only", "randproj_iom")
ARM_NOTE = {
    "polyiom": "z -> P_K*(z; o*) -> IoM-GRP   (the proposed method)",
    "iom_only": "z ----------------> IoM-GRP   (the practical baseline)",
    "randproj_iom": "z -> A z ----------> IoM-GRP   (isolates the polynomial)",
}


def abl_seed(modality, label):
    """Deterministic seed, distinct from every seed stream in the study."""
    return H(S0, "ABLATION", modality, label)


def abl_randn(shape, modality, label):
    generator = torch.Generator(device="cpu")
    generator.manual_seed(abl_seed(modality, label))
    return torch.randn(shape, generator=generator, dtype=torch.float32)


def hash_arm(vectors, arm, key, overlap, tensor, M, q, A, batch_size=64):
    """One arm's protected codes. Only the pre-projection step differs."""
    coefficients, exponents = key
    tensor = tensor[:M, :q].to(DEVICE, torch.float32)
    outputs = []
    for start in range(0, len(vectors), batch_size):
        batch = torch.as_tensor(
            np.stack(vectors[start:start + batch_size]),
            dtype=torch.float32, device=DEVICE,
        )
        if arm == "polyiom":
            projected = poly_transform(batch, coefficients, exponents, overlap)
        elif arm == "randproj_iom":
            projected = batch @ A
        elif arm == "iom_only":
            projected = batch
        else:
            raise ValueError(arm)
        scores = torch.einsum("bk,mqk->bmq", projected, tensor)
        outputs.append(torch.argmax(scores, dim=-1).to(torch.int16).cpu().numpy())
    return np.concatenate(outputs, axis=0)


def check_gaussian(tensor, label, tolerance=0.05):
    """The generated tensors are standard normal. If the frozen recognition
    tensor is not, the arms are not drawn from the same family and the
    comparison is confounded, so say so loudly rather than quietly."""
    mean, std = float(tensor.mean()), float(tensor.std())
    ok = abs(mean) <= tolerance and abs(std - 1.0) <= tolerance
    print(f"    {label}: mean={mean:+.4f} std={std:.4f} "
          f"{'~N(0,1)' if ok else '*** NOT ~N(0,1) - comparison confounded'}")
    return ok


def build_arm_histograms(modality):
    """Per-identity histograms for all three arms, sharing identity order."""
    sealed = json.loads(HELDOUT_RESULTS[modality].read_text())
    M, q, overlap = sealed["M"], sealed["q"], sealed["o"]
    threshold = int(sealed["c_tau_carried_from_DEV"])
    subjects, enroll, probes = (
        face_heldout_data() if modality == "face" else voice_heldout_data()
    )
    if len(subjects) != sealed["n_subjects"]:
        raise RuntimeError(
            f"{modality}: reconstructed {len(subjects)} identities; "
            f"sealed result has {sealed['n_subjects']}"
        )

    key = selected_poly_key(modality)
    n = len(subjects)
    d = len(enroll[subjects[0]])
    k = 1 + math.ceil((d - G) / (G - overlap))
    print(f"  {modality}: n={n}  d={d}  k={k}  M={M} q={q} o={overlap}")

    frozen = load_iom_tensor(modality, overlap, d)
    print("  recognition tensors:")
    gaussian_ok = check_gaussian(frozen[:M, :q], "polyiom (frozen, from disk)")
    recognition = {
        "polyiom": frozen,
        "iom_only": abl_randn((M, q, d), modality, "match|iom_only"),
        "randproj_iom": abl_randn((M, q, k), modality, "match|randproj"),
    }
    A = abl_randn((d, k), modality, "projmap") / math.sqrt(d)

    unlink = {"polyiom": [], "iom_only": [], "randproj_iom": []}
    for key_index in (1, 2):
        generator = torch.Generator(device="cpu")
        generator.manual_seed(seed_unlink(modality, key_index))
        unlink["polyiom"].append(
            torch.randn((M, q, k), generator=generator, dtype=torch.float32)
        )
        unlink["iom_only"].append(
            abl_randn((M, q, d), modality, f"unlink|raw|{key_index}")
        )
        unlink["randproj_iom"].append(
            abl_randn((M, q, k), modality, f"unlink|prj|{key_index}")
        )

    enrolment_vectors = [enroll[uid] for uid in subjects]
    probe_vectors, probe_owners = [], []
    for owner, uid in enumerate(subjects):
        probe_vectors.extend(probes[uid])
        probe_owners.extend([owner] * len(probes[uid]))

    bundles = {}
    for arm in ARMS:
        enrolment_hashes = hash_arm(
            enrolment_vectors, arm, key, overlap, recognition[arm], M, q, A)
        probe_hashes = hash_arm(
            probe_vectors, arm, key, overlap, recognition[arm], M, q, A)
        collision = (probe_hashes[:, None, :]
                     == enrolment_hashes[None, :, :]).sum(axis=-1)

        genuine_by_subject = np.zeros((n, M + 1), dtype=np.float64)
        impostor_by_pair = np.zeros((n, n, M + 1), dtype=np.float64)
        for probe_index, owner in enumerate(probe_owners):
            genuine_by_subject[owner, collision[probe_index, owner]] += 1
            for claimant in range(n):
                if claimant != owner:
                    impostor_by_pair[owner, claimant,
                                     collision[probe_index, claimant]] += 1

        first = hash_arm(enrolment_vectors, arm, key, overlap,
                         unlink[arm][0], M, q, A)
        second = hash_arm(enrolment_vectors, arm, key, overlap,
                          unlink[arm][1], M, q, A)
        unlink_collision = (first[:, None, :] == second[None, :, :]).sum(axis=-1)
        mated_by_subject = np.zeros((n, M + 1), dtype=np.float64)
        nonmated_by_pair = np.zeros((n, n, M + 1), dtype=np.float64)
        for left in range(n):
            mated_by_subject[left, unlink_collision[left, left]] = 1
            for right in range(n):
                if left != right:
                    nonmated_by_pair[left, right,
                                     unlink_collision[left, right]] = 1

        bundles[arm] = {
            "genuine_by_subject": genuine_by_subject,
            "impostor_by_pair": impostor_by_pair,
            "mated_by_subject": mated_by_subject,
            "nonmated_by_pair": nonmated_by_pair,
        }
        print(f"    {arm:<13} built   {ARM_NOTE[arm]}")

    return {
        "modality": modality, "sealed": sealed, "subjects": subjects,
        "M": M, "q": q, "o": overlap, "d": d, "k": k,
        "threshold": threshold, "arms": bundles,
        "frozen_tensor_is_gaussian": bool(gaussian_ok),
    }


print("Arm construction ready.")


def eer_from_histograms(genuine, impostor):
    genuine_total, impostor_total = genuine.sum(), impostor.sum()
    if genuine_total <= 0 or impostor_total <= 0:
        raise RuntimeError("EER requires non-empty distributions")
    far = np.r_[np.cumsum(impostor[::-1])[::-1], 0.0] / impostor_total
    frr = np.r_[0.0, np.cumsum(genuine)] / genuine_total
    difference = far - frr
    c_eer = int(np.flatnonzero(np.abs(difference) == np.abs(difference).min())[0])
    exact = np.flatnonzero(difference == 0.0)
    if len(exact):
        return float(far[exact[0]]), c_eer
    crossings = np.flatnonzero((difference[:-1] > 0.0) & (difference[1:] < 0.0))
    if not len(crossings):
        raise AssertionError("EER crossing not found")
    index = int(crossings[0])
    weight = difference[index] / (difference[index] - difference[index + 1])
    return float(far[index] + weight * (far[index + 1] - far[index])), c_eer


def dsys_from_histograms(mated, nonmated):
    if mated.sum() <= 0 or nonmated.sum() <= 0:
        raise RuntimeError("Dsys requires non-empty distributions")
    pm, pn = mated / mated.sum(), nonmated / nonmated.sum()
    local = np.zeros_like(pm)
    for score in range(len(pm)):
        if pm[score] == 0:
            local[score] = 0
        elif pn[score] == 0:
            local[score] = 1
        else:
            ratio = pm[score] / pn[score]
            local[score] = 0 if ratio <= 1 else 2 * ratio / (1 + ratio) - 1
    return float(np.sum(pm * local))


def collapse(bundle, identity_weights, n):
    pair_weights = np.outer(identity_weights, identity_weights)
    np.fill_diagonal(pair_weights, 0.0)
    flat = pair_weights.reshape(-1)
    return (
        identity_weights @ bundle["genuine_by_subject"],
        flat @ bundle["impostor_by_pair"].reshape(n * n, -1),
        identity_weights @ bundle["mated_by_subject"],
        flat @ bundle["nonmated_by_pair"].reshape(n * n, -1),
    )


def arm_metrics(bundle, identity_weights, n):
    genuine, impostor, mated, nonmated = collapse(bundle, identity_weights, n)
    eer, c_eer = eer_from_histograms(genuine, impostor)
    return {"EER": eer, "c_EER": c_eer,
            "Dsys": dsys_from_histograms(mated, nonmated)}, genuine, impostor


def tar_at_matched_fmr(genuine, impostor, target):
    """Lowest threshold whose FMR still meets the target; descriptive only."""
    total = impostor.sum()
    best = None
    for c in range(len(impostor) - 1, -1, -1):
        if float(impostor[c:].sum() / total) <= target + 1e-12:
            best = c
        else:
            break
    if best is None:
        return float("nan"), -1
    return float(genuine[best:].sum() / genuine.sum()), best


def verify_against_seal(record):
    """The polyiom arm rebuilds the sealed pipeline exactly, so its EER must
    reproduce the sealed held-out figure. Anything else means this notebook
    has drifted from the study and its numbers must not be reported."""
    sealed = record["sealed"]
    reproduced = record["point"]["polyiom"]["EER"]
    if not np.isclose(reproduced, sealed["EER_HOLDOUT"], rtol=0.0, atol=1e-12):
        raise RuntimeError(
            f"{record['modality']}: the polyiom arm reproduces "
            f"EER={reproduced}, but the sealed held-out result is "
            f"{sealed['EER_HOLDOUT']}. Refusing to report - the arm is not "
            f"the study's pipeline."
        )
    print(f"  self-check: polyiom EER {reproduced * 100:.4f}% reproduces the "
          f"sealed held-out result")


def paired_bootstrap(record, n_bootstrap, confidence, seed):
    """One identity weight vector per replicate, applied to all three arms,
    so an interval on a DIFFERENCE accounts for the arms sharing identities.
    An unpaired interval would be too wide."""
    n = len(record["subjects"])
    rng = np.random.default_rng(seed)
    parts, remaining = [], n_bootstrap
    while remaining:
        candidates = rng.multinomial(n, np.full(n, 1 / n), size=remaining)
        candidates = candidates[(candidates > 0).sum(axis=1) >= 2]
        parts.append(candidates[:remaining])
        remaining -= len(parts[-1])
    weights = np.concatenate(parts, axis=0).astype(np.float64)

    per_arm = {arm: {"EER": [], "Dsys": []} for arm in ARMS}
    contrasts = {arm: {"delta_EER": [], "delta_Dsys": []}
                 for arm in ARMS if arm != "polyiom"}
    for replicate in range(n_bootstrap):
        w = weights[replicate]
        values = {}
        for arm in ARMS:
            measured, _, _ = arm_metrics(record["arms"][arm], w, n)
            values[arm] = measured
            per_arm[arm]["EER"].append(measured["EER"])
            per_arm[arm]["Dsys"].append(measured["Dsys"])
        for arm in contrasts:
            contrasts[arm]["delta_EER"].append(
                values[arm]["EER"] - values["polyiom"]["EER"])
            contrasts[arm]["delta_Dsys"].append(
                values[arm]["Dsys"] - values["polyiom"]["Dsys"])
        if (replicate + 1) % 500 == 0:
            print(f"    {record['modality']}: {replicate + 1}/{n_bootstrap}")

    alpha = 1.0 - confidence

    def interval(values):
        lower, upper = np.quantile(values, [alpha / 2, 1 - alpha / 2])
        return [float(lower), float(upper)]

    return per_arm, contrasts, interval


def run_ablation(n_bootstrap=2000, confidence=0.95):
    if ABLATION_OUTPUT.is_file():
        stored = json.loads(ABLATION_OUTPUT.read_text())
        print("Resume: stored ablation result found; returning it unchanged.")
        return stored

    source_hashes = {m: sha256_file(p) for m, p in HELDOUT_RESULTS.items()}
    result = {
        "schema": 1,
        "analysis": "hardening_ablation",
        "method_id": "identity_cluster_percentile_v1_paired",
        "n_bootstrap": int(n_bootstrap),
        "confidence": float(confidence),
        "master_seed": S0,
        "changes_any_seal": False,
        "threshold_policy": "none fitted; EER and Dsys are threshold-free",
        "arms": {arm: ARM_NOTE[arm] for arm in ARMS},
        "source_heldout_sha256": source_hashes,
        "modalities": {},
    }

    for modality in ("voice", "face"):
        print(f"\n{modality.upper()}")
        record = build_arm_histograms(modality)
        n = len(record["subjects"])
        flat = np.ones(n, dtype=np.float64)

        record["point"] = {}
        matched = {}
        for arm in ARMS:
            measured, genuine, impostor = arm_metrics(record["arms"][arm], flat, n)
            record["point"][arm] = measured
            matched[arm] = (genuine, impostor)
        verify_against_seal(record)

        reference_fmr = float(
            matched["polyiom"][1][record["threshold"]:].sum()
            / matched["polyiom"][1].sum()
        )
        for arm in ARMS:
            tar, c = tar_at_matched_fmr(*matched[arm], reference_fmr)
            record["point"][arm]["TAR_at_matched_FMR_descriptive"] = tar
            record["point"][arm]["c_matched"] = int(c)

        print(f"  paired bootstrap over {n} identities")
        per_arm, contrasts, interval = paired_bootstrap(
            record, n_bootstrap, confidence,
            seed=abl_seed(modality, "bootstrap"),
        )

        result["modalities"][modality] = {
            "n_subjects": n, "M": record["M"], "q": record["q"],
            "o": record["o"], "d": record["d"], "k": record["k"],
            "c_tau_carried_from_DEV": record["threshold"],
            "frozen_tensor_is_gaussian": record["frozen_tensor_is_gaussian"],
            "polyiom_reproduces_sealed_heldout": True,
            "reference_FMR_for_matched_TAR": reference_fmr,
            "arms": {arm: {
                "EER": record["point"][arm]["EER"],
                "EER_CI": interval(per_arm[arm]["EER"]),
                "Dsys": record["point"][arm]["Dsys"],
                "Dsys_CI": interval(per_arm[arm]["Dsys"]),
                "TAR_at_matched_FMR_descriptive":
                    record["point"][arm]["TAR_at_matched_FMR_descriptive"],
            } for arm in ARMS},
            "contrasts_vs_polyiom": {arm: {
                "delta_EER": (record["point"][arm]["EER"]
                              - record["point"]["polyiom"]["EER"]),
                "delta_EER_CI": interval(contrasts[arm]["delta_EER"]),
                "delta_Dsys": (record["point"][arm]["Dsys"]
                               - record["point"]["polyiom"]["Dsys"]),
                "delta_Dsys_CI": interval(contrasts[arm]["delta_Dsys"]),
            } for arm in contrasts},
        }

    if {m: sha256_file(p) for m, p in HELDOUT_RESULTS.items()} != source_hashes:
        raise RuntimeError("A sealed held-out result changed during the ablation")
    atomic_json(ABLATION_OUTPUT, result)
    print("\nWrote", ABLATION_OUTPUT)
    return result


def ablation_table(result):
    rows = []
    for modality, data in result["modalities"].items():
        for arm, values in data["arms"].items():
            rows.append({
                "modality": modality, "arm": arm,
                "EER": values["EER"],
                "EER_lo": values["EER_CI"][0], "EER_hi": values["EER_CI"][1],
                "Dsys": values["Dsys"],
                "Dsys_lo": values["Dsys_CI"][0], "Dsys_hi": values["Dsys_CI"][1],
                "TAR_matched_FMR": values["TAR_at_matched_FMR_descriptive"],
            })
    return pd.DataFrame(rows)


def contrast_report(result):
    for modality, data in result["modalities"].items():
        print(f"\n{modality.upper()}  d={data['d']} -> k={data['k']}  "
              f"n={data['n_subjects']} identities")
        print("  arm minus proposed; an interval excluding 0 is firm")
        for arm, c in data["contrasts_vs_polyiom"].items():
            for name, scale, unit in (("EER", 100, " %"), ("Dsys", 1, "")):
                value = c[f"delta_{name}"]
                lower, upper = c[f"delta_{name}_CI"]
                firm = "FIRM" if (lower > 0 or upper < 0) else "not distinguishable"
                print(f"    {arm:<14} d{name:<5}{value * scale:+8.3f}{unit}"
                      f"  [{lower * scale:+7.3f},{upper * scale:+7.3f}]  {firm}")
    print("\n  A POSITIVE delta means that arm is WORSE than the proposed")
    print("  method (higher EER, higher Dsys). randproj_iom is the arm that")
    print("  isolates the polynomial: it matches the proposed method's")
    print("  projection dimension k, so a null contrast there means the gain")
    print("  came from the dimension change and not from P_K*.")


print("Metrics and paired bootstrap ready.")


In [ ]:
%%writefile {RUNTIME}/scoredump_only.py
# 3. The study's score histograms
"""Score-histogram dump for AttackAware PolyIoM v1.1.4.

Loaded after ablation_only.py, which supplies the frozen-pipeline
primitives. Read-only: writes one file and no seal.

Why histograms rather than raw score vectors
--------------------------------------------
A collision score is an integer in 0..M. A count per score value is
therefore a LOSSLESS record of the score distribution, and every results
figure a biometric paper needs can be derived from it exactly:

  * FMR and FNMR at every threshold, hence the full DET curve
  * EER, and the threshold at which it occurs
  * genuine and impostor distributions, with tau* marked
  * mated and non-mated distributions, which is what D_sys summarises

The file is a few kilobytes rather than a few megabytes, and nothing is
approximated. The evaluation already builds these histograms internally
and discards them; this cell keeps them.

Covers two systems, because the first criterion a cancelable-biometric
paper has to address is performance PRESERVATION, which is reported as the
protected system against the UNPROTECTED one:

  protected     the sealed pipeline, integer collision scores in 0..M
  unprotected   plain cosine similarity between the same embeddings

Cosine is continuous, so the unprotected side is binned finely over
[-1, 1] and the bin edges are stored with it. At 2000 bins the resolution
is 0.001, which is far finer than any DET curve needs.

Also stores the LOCAL unlinkability curve D_link(s) alongside the global
value. The global value is what the evaluation already reports; the local
curve is the part that is plottable, and it is the standard unlinkability
figure in this literature.

Note that the mated distribution serves two figures at once. Under the
unlinkability framework it is "same subject, two different keys"; under
revocability it is the pseudo-impostor distribution. They are the same
measurement, and a paper should not present them as independent evidence.

Partitions: development and evaluation for both modalities, plus external
VCTK for voice when its embeddings are present.
"""

COS_BINS = 2000

SCORE_OUTPUT = None          # set in run_score_dump


def partition_data(modality, partition):
    """Enrolment and probe embeddings for one partition of one modality.

    Same construction as the held-out loaders in ablation_only.py, with the
    partition as a parameter so development can be read the same way.
    """
    if modality == "face":
        trials = pd.read_csv(LFW_FACE_TRIALS, sep="\t")
        stored = np.load(LFW_EMB, allow_pickle=True)
        embeddings = stored["embedding"].astype(np.float32)
        rows = trials[trials.partition == partition]
        subjects, enroll, probes = [], {}, {}
        for uid, group in rows.groupby("subject_id"):
            erow = group[group.role == "enroll"]
            prows = group[group.role == "probe"]
            if len(erow) != 1 or len(prows) < 1:
                continue
            uid = str(uid)
            subjects.append(uid)
            enroll[uid] = embeddings[int(erow.iloc[0].embedding_index)]
            probes[uid] = [embeddings[int(r.embedding_index)]
                           for r in prows.itertuples()]
        return sorted(subjects), enroll, probes

    manifest = pd.read_csv(LIBRI_MANIFEST, sep="\t", dtype={"speaker_id": str})
    stored = np.load(LIBRI_EMB, allow_pickle=True)
    embeddings = stored["embedding"].astype(np.float32)
    table = pd.DataFrame({
        "speaker_id": stored["speaker_id"].astype(str),
        "rank": stored["rank"].astype(int),
        "embedding_index": np.arange(len(embeddings), dtype=np.int64),
    })
    rows = manifest.merge(table, on=["speaker_id", "rank"], how="inner")
    rows = rows[rows.partition == partition]
    subjects, enroll, probes = [], {}, {}
    for uid, group in rows.groupby("speaker_id"):
        group = group.sort_values("rank")
        if len(group) != 15:
            continue
        uid = str(uid)
        members = np.stack([embeddings[int(r.embedding_index)]
                            for r in group[group["rank"] < 5].itertuples()])
        subjects.append(uid)
        enroll[uid] = l2_np(members.mean(axis=0))
        probes[uid] = [embeddings[int(r.embedding_index)]
                       for r in group[group["rank"] >= 5].itertuples()]
    return sorted(subjects), enroll, probes


def histograms_for(modality, subjects, enroll, probes, M, q, overlap, key,
                   recognition, unlink_pair):
    """Flat genuine / impostor / mated / non-mated histograms over 0..M."""
    enrolment = [enroll[uid] for uid in subjects]
    enrolled = hash_arm(enrolment, "polyiom", key, overlap, recognition,
                        M, q, None)
    probe_vectors, owners = [], []
    for index, uid in enumerate(subjects):
        probe_vectors.extend(probes[uid])
        owners.extend([index] * len(probes[uid]))
    hashed = hash_arm(probe_vectors, "polyiom", key, overlap, recognition,
                      M, q, None)
    collision = (hashed[:, None, :] == enrolled[None, :, :]).sum(axis=-1)

    genuine = np.zeros(M + 1, dtype=np.int64)
    impostor = np.zeros(M + 1, dtype=np.int64)
    for row, owner in enumerate(owners):
        for claimant in range(len(subjects)):
            c = int(collision[row, claimant])
            if claimant == owner:
                genuine[c] += 1
            else:
                impostor[c] += 1

    first = hash_arm(enrolment, "polyiom", key, overlap, unlink_pair[0],
                     M, q, None)
    second = hash_arm(enrolment, "polyiom", key, overlap, unlink_pair[1],
                      M, q, None)
    cross = (first[:, None, :] == second[None, :, :]).sum(axis=-1)
    mated = np.zeros(M + 1, dtype=np.int64)
    nonmated = np.zeros(M + 1, dtype=np.int64)
    for a in range(len(subjects)):
        mated[int(cross[a, a])] += 1
        for b in range(len(subjects)):
            if a != b:
                nonmated[int(cross[a, b])] += 1

    return {"genuine": genuine.tolist(), "impostor": impostor.tolist(),
            "mated": mated.tolist(), "nonmated": nonmated.tolist(),
            "n_subjects": len(subjects),
            "n_genuine": int(genuine.sum()),
            "n_impostor": int(impostor.sum())}


def cosine_histograms(subjects, enroll, probes, nbins=COS_BINS):
    """Unprotected baseline: plain cosine between the same embeddings.

    This is the reference the protected system has to be compared against.
    Binned over [-1, 1] rather than stored raw: 2000 bins is 0.001
    resolution and a few kilobytes instead of megabytes.
    """
    def unit(a):
        a = np.asarray(a, dtype=np.float64)
        return a / max(np.linalg.norm(a), 1e-12)

    E = np.stack([unit(enroll[uid]) for uid in subjects])
    rows, owners = [], []
    for index, uid in enumerate(subjects):
        for v in probes[uid]:
            rows.append(unit(v))
            owners.append(index)
    P = np.stack(rows)
    scores = P @ E.T                                  # (probes, subjects)

    edges = np.linspace(-1.0, 1.0, nbins + 1)
    own = np.asarray(owners)
    mask = np.zeros_like(scores, dtype=bool)
    mask[np.arange(len(own)), own] = True
    gen, _ = np.histogram(scores[mask], bins=edges)
    imp, _ = np.histogram(scores[~mask], bins=edges)
    return {"genuine": gen.tolist(), "impostor": imp.tolist(),
            "bin_edges": [float(edges[0]), float(edges[-1])],
            "n_bins": int(nbins),
            "n_genuine": int(gen.sum()), "n_impostor": int(imp.sum())}


def unlinkability_curve(mated, nonmated):
    """Local D_link(s) and the global value, from the same histograms.

    The global value is what the evaluation already reports. The local
    curve is what a figure needs.
    """
    m = np.asarray(mated, float)
    n = np.asarray(nonmated, float)
    if m.sum() <= 0 or n.sum() <= 0:
        return [], float("nan")
    pm, pn = m / m.sum(), n / n.sum()
    local = np.zeros_like(pm)
    for s_ in range(len(pm)):
        if pm[s_] == 0:
            local[s_] = 0.0
        elif pn[s_] == 0:
            local[s_] = 1.0
        else:
            ratio = pm[s_] / pn[s_]
            local[s_] = 0.0 if ratio <= 1 else 2 * ratio / (1 + ratio) - 1
    return local.tolist(), float(np.sum(pm * local))


def eer_cosine(hist_gen, hist_imp):
    """EER from a binned continuous score, for the unprotected baseline.

    The exact-zero branch is not optional here. When the two distributions
    are well separated there is a run of thresholds at which FAR and FRR
    are both zero, so FAR - FRR never changes sign between adjacent bins
    and a crossing-only search returns nothing. The unprotected baseline
    is the case most likely to be well separated, which is exactly the
    number this function exists to produce.
    """
    g = np.asarray(hist_gen, float)
    i = np.asarray(hist_imp, float)
    if g.sum() <= 0 or i.sum() <= 0:
        return float("nan")
    far = np.r_[np.cumsum(i[::-1])[::-1], 0.0] / i.sum()
    frr = np.r_[0.0, np.cumsum(g)] / g.sum()
    d = far - frr
    exact = np.flatnonzero(d == 0.0)
    if len(exact):
        return float(far[exact[0]])
    cross = np.flatnonzero((d[:-1] > 0) & (d[1:] < 0))
    if len(cross):
        j = int(cross[0])
        w = d[j] / (d[j] - d[j + 1])
        return float(far[j] + w * (far[j + 1] - far[j]))
    j = int(np.argmin(np.maximum(far, frr)))      # never crosses
    return float((far[j] + frr[j]) / 2.0)


def eer_check(genuine, impostor, M):
    """Recompute EER from the histogram, to prove the dump is faithful."""
    g = np.asarray(genuine, float)
    i = np.asarray(impostor, float)
    far = np.r_[np.cumsum(i[::-1])[::-1], 0.0] / i.sum()
    frr = np.r_[0.0, np.cumsum(g)] / g.sum()
    d = far - frr
    exact = np.flatnonzero(d == 0.0)
    if len(exact):
        return float(far[exact[0]])
    cross = np.flatnonzero((d[:-1] > 0) & (d[1:] < 0))
    j = int(cross[0])
    w = d[j] / (d[j] - d[j + 1])
    return float(far[j] + w * (far[j + 1] - far[j]))


def run_score_dump():
    output = DIR["runs"] / "scores" / "score_histograms.json"
    if output.is_file():
        print("Resume: stored score histograms found; returning unchanged.")
        return json.loads(output.read_text())

    result = {"schema": 1, "analysis": "score_histograms",
              "systems": {
                  "protected": "polyiom, the sealed pipeline; integer "
                               "collision scores in 0..M",
                  "unprotected": "plain cosine between the same embeddings; "
                                 "binned over [-1,1], the performance-"
                                 "preservation reference"},
              "mated_note": "the mated distribution is both the "
                            "unlinkability mated set and the revocability "
                            "pseudo-impostor set; one measurement, not two",
              "lossless": "scores are integers in 0..M, so a count per "
                          "score value loses nothing",
              "master_seed": S0, "changes_any_seal": False, "modalities": {}}

    for modality in ("voice", "face"):
        sealed = json.loads(HELDOUT_RESULTS[modality].read_text())
        M, q, overlap = sealed["M"], sealed["q"], sealed["o"]
        tau = int(sealed["c_tau_carried_from_DEV"])
        key = selected_poly_key(modality)
        print(f"\n{modality.upper()}  M={M} q={q} o={overlap} tau={tau}")

        entry = {"M": M, "q": q, "o": overlap, "tau": tau,
                 "partitions": {}}
        for partition in ("development", "evaluation"):
            subjects, enroll, probes = partition_data(modality, partition)
            if len(subjects) < 2:
                print(f"  {partition}: too few identities, skipped")
                continue
            d = len(enroll[subjects[0]])
            k = 1 + math.ceil((d - G) / (G - overlap))
            recognition = load_iom_tensor(modality, overlap, d)[:M, :q] \
                .to(DEVICE, torch.float32)
            pair = []
            for index in (1, 2):
                gen = torch.Generator(device="cpu")
                gen.manual_seed(seed_unlink(modality, index))
                pair.append(torch.randn((M, q, k), generator=gen,
                                        dtype=torch.float32).to(DEVICE))
            h = histograms_for(modality, subjects, enroll, probes, M, q,
                               overlap, key, recognition, pair)
            h["dlink_curve"], h["dlink_global"] = unlinkability_curve(
                h["mated"], h["nonmated"])
            h["unprotected"] = cosine_histograms(subjects, enroll, probes)
            entry["partitions"][partition] = h
            recomputed = eer_check(h["genuine"], h["impostor"], M)
            raw = eer_cosine(h["unprotected"]["genuine"],
                             h["unprotected"]["impostor"])
            h["unprotected"]["EER"] = raw
            note = ""
            if partition == "evaluation":
                delta = abs(recomputed - sealed["EER_HOLDOUT"])
                note = (f"  reproduces sealed held-out EER "
                        f"({sealed['EER_HOLDOUT'] * 100:.4f}%)"
                        if delta <= 1e-9 else
                        f"  *** MISMATCH vs sealed "
                        f"{sealed['EER_HOLDOUT'] * 100:.4f}%")
            print(f"  {partition:<12} {h['n_subjects']} identities, "
                  f"{h['n_genuine']} genuine / {h['n_impostor']} impostor")
            print(f"    protected   EER {recomputed * 100:7.4f}%{note}")
            print(f"    unprotected EER {raw * 100:7.4f}%   "
                  f"-> protection costs "
                  f"{(recomputed - raw) * 100:+.4f} pp")
            print(f"    D_link global {h['dlink_global']:.6f}")
        result["modalities"][modality] = entry

    # External voice, if its embeddings are present
    ext = PROJECT / "external_voice_only.py"
    vctk = DIR["embeddings"] / "vctk_external_embeddings.npz"
    if ext.is_file() and vctk.is_file():
        try:
            scope = {"PROJECT": PROJECT, "DIR": DIR, "np": np, "pd": pd,
                     "torch": torch, "json": json, "math": math,
                     "hashlib": hashlib, "os": os, "Path": Path}
            exec(compile(ext.read_text(), str(ext), "exec"), scope)
            subjects, enroll, probes = scope["external_voice_data"]()
            sealed = json.loads(HELDOUT_RESULTS["voice"].read_text())
            M, q, overlap = sealed["M"], sealed["q"], sealed["o"]
            key = selected_poly_key("voice")
            d = len(enroll[subjects[0]])
            k = 1 + math.ceil((d - G) / (G - overlap))
            recognition = load_iom_tensor("voice", overlap, d)[:M, :q] \
                .to(DEVICE, torch.float32)
            pair = []
            for index in (1, 2):
                gen = torch.Generator(device="cpu")
                gen.manual_seed(seed_unlink("voice", index))
                pair.append(torch.randn((M, q, k), generator=gen,
                                        dtype=torch.float32).to(DEVICE))
            h = histograms_for("voice", subjects, enroll, probes, M, q,
                               overlap, key, recognition, pair)
            h["dlink_curve"], h["dlink_global"] = unlinkability_curve(
                h["mated"], h["nonmated"])
            h["unprotected"] = cosine_histograms(subjects, enroll, probes)
            h["unprotected"]["EER"] = eer_cosine(
                h["unprotected"]["genuine"], h["unprotected"]["impostor"])
            result["modalities"]["voice"]["partitions"]["external"] = h
            print(f"\n  external     {h['n_subjects']} speakers, "
                  f"{h['n_genuine']} genuine / {h['n_impostor']} impostor")
            print(f"    protected   EER "
                  f"{eer_check(h['genuine'], h['impostor'], M) * 100:7.4f}%")
            print(f"    unprotected EER "
                  f"{h['unprotected']['EER'] * 100:7.4f}%")
        except Exception as exc:
            print(f"\n  external: skipped - {type(exc).__name__}: {exc}")
    else:
        print("\n  external: skipped - VCTK embeddings or runtime not found")

    atomic_json(output, result)
    print("\nWrote", output)
    return result


print("Score dump ready. Run run_score_dump().")


In [ ]:
%%writefile {RUNTIME}/inversion_only.py
# 4. The study's full-knowledge inversion attack
"""Full-knowledge inversion evaluation for AttackAware PolyIoM v1.1.4.

Loaded by AttackAware_PolyIoM_v1_1_4_INVERSION.ipynb after its preflight
cell, and after ablation_only.py has supplied the frozen-pipeline
primitives. Pre-specification is in THREAT_MODEL.md; budget and success
criteria were fixed before this was run.

Writes one file, runs/inversion/inversion_result.json, and no seal.
"""

INV_RESTARTS = 5
INV_STEPS = 800
INV_LR = 0.05
INV_TEMP0, INV_TEMP1 = 1.0, 0.05
INV_N_BOOTSTRAP = 2000
INV_CONFIDENCE = 0.95


def poly_transform_diff(vectors, coefficients, exponents, overlap):
    """Differentiable polynomial, numerically identical to the study's.

    The study calls torch.pow(values, int64_tensor). Casting that exponent
    to float to get gradients would produce nan on negative bases, so the
    powers are built with Python integer exponents instead, which keeps
    integer-power semantics and a defined gradient.
    """
    idx, mask = poly_indices(vectors.shape[-1], overlap, vectors.device)
    values = vectors[..., idx] * mask.to(vectors.dtype)
    coefficients = torch.as_tensor(
        coefficients, device=vectors.device, dtype=vectors.dtype)
    powers = torch.stack(
        [values[..., i] ** int(exponents[i]) for i in range(values.shape[-1])],
        dim=-1,
    )
    return (powers * coefficients).sum(dim=-1)


def assert_diff_matches(modality, key, overlap, d, tolerance=1e-4):
    """Assertion 1 - the differentiable path is the study's path.

    Checked on RELATIVE error, and on the bucket indices it produces, not
    on absolute error. torch.pow(x, int64_tensor) and x ** int dispatch to
    different kernels, so they round differently; on a polynomial whose
    intermediate values can be large that shows as an absolute gap of a few
    1e-4 while the relative gap is ~1e-7. An earlier absolute threshold of
    1e-4 here was simply the wrong test.

    What actually matters is that no bucket index moves. This function
    checks that directly on the probes, and assertion 2 then confirms it
    end to end against the real stored templates.
    """
    generator = torch.Generator(device="cpu")
    generator.manual_seed(H(S0, "INVERSION-CHECK", modality))
    probe = torch.randn((64, d), generator=generator, dtype=torch.float32)
    probe = probe / probe.norm(dim=-1, keepdim=True)   # as real embeddings are
    reference = poly_transform(probe, key[0], key[1], overlap)
    candidate = poly_transform_diff(probe, key[0], key[1], overlap)

    absolute = float((reference - candidate).abs().max())
    scale = float(reference.abs().max())
    relative = absolute / max(scale, 1e-30)
    if not relative < tolerance:
        raise RuntimeError(
            f"{modality}: differentiable polynomial differs from the "
            f"study's by a relative {relative:.3e} (absolute {absolute:.3e} "
            f"at scale {scale:.3e}); the attack would not be attacking the "
            f"real scheme."
        )

    # The decisive check: same argmax, on a tensor of the right width.
    k = 1 + math.ceil((d - G) / (G - overlap))
    probe_generator = torch.Generator(device="cpu")
    probe_generator.manual_seed(H(S0, "INVERSION-CHECK-TENSOR", modality))
    probe_tensor = torch.randn((64, 32, k), generator=probe_generator,
                               dtype=torch.float32)
    left = torch.argmax(torch.einsum("bk,mqk->bmq", reference, probe_tensor), -1)
    right = torch.argmax(torch.einsum("bk,mqk->bmq", candidate, probe_tensor), -1)
    moved = int((left != right).sum())
    if moved:
        raise RuntimeError(
            f"{modality}: {moved} bucket indices move between the study's "
            f"polynomial and the differentiable one; the attack would not "
            f"be attacking the real scheme."
        )
    print(f"    assertion 1 OK: differentiable polynomial matches "
          f"(relative {relative:.2e}, absolute {absolute:.2e} at scale "
          f"{scale:.2e}) and moves no bucket index")


def project(arm, z, key, overlap, A):
    """Arm-specific pre-projection, rescaled to unit norm.

    argmax within a group is invariant to a positive per-sample rescale, so
    this changes no bucket index (assertion 2 checks that empirically). It
    exists so the annealed temperature means the same thing for every arm:
    the polynomial's output norm is not 1, and without this the three arms
    would be attacked at different effective temperatures, which would make
    the comparison unfair and could overflow the softmax.
    """
    if arm == "polyiom":
        x = poly_transform_diff(z, key[0], key[1], overlap)
    elif arm == "randproj_iom":
        x = z @ A
    elif arm == "iom_only":
        x = z
    else:
        raise ValueError(arm)
    return x / x.norm(dim=-1, keepdim=True).clamp_min(1e-12)


def codes_of(arm, z, key, overlap, tensor, A):
    """Hard bucket indices, no gradient."""
    with torch.no_grad():
        scores = torch.einsum(
            "bk,mqk->bmq", project(arm, z, key, overlap, A), tensor)
        return torch.argmax(scores, dim=-1)


def invert_arm(arm, targets, key, overlap, tensor, A, d, seed,
               restarts=INV_RESTARTS, steps=INV_STEPS, lr=INV_LR):
    """Attack every template at once. Identical budget for every arm."""
    n = targets.shape[0]
    best_z = torch.zeros((n, d), dtype=torch.float32, device=DEVICE)
    best_hits = torch.full((n,), -1, dtype=torch.int64, device=DEVICE)

    for restart in range(restarts):
        generator = torch.Generator(device="cpu")
        generator.manual_seed(seed + restart)
        z = torch.randn((n, d), generator=generator, dtype=torch.float32)
        z = (z / z.norm(dim=-1, keepdim=True)).to(DEVICE).requires_grad_(True)
        optimiser = torch.optim.Adam([z], lr=lr)

        for step in range(steps):
            temperature = INV_TEMP0 * (INV_TEMP1 / INV_TEMP0) ** (step / max(steps - 1, 1))
            scores = torch.einsum(
                "bk,mqk->bmq", project(arm, z, key, overlap, A), tensor)
            loss = torch.nn.functional.cross_entropy(
                (scores / temperature).reshape(-1, scores.shape[-1]),
                targets.reshape(-1),
            )
            optimiser.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_([z], 5.0)
            optimiser.step()
            with torch.no_grad():
                z /= z.norm(dim=-1, keepdim=True).clamp_min(1e-12)

        hits = (codes_of(arm, z.detach(), key, overlap, tensor, A)
                == targets).sum(dim=-1)
        improved = hits > best_hits
        best_hits = torch.where(improved, hits, best_hits)
        best_z[improved] = z.detach()[improved]
        print(f"      {arm:<13} restart {restart + 1}/{restarts}  "
              f"loss {float(loss.detach()):7.4f}  best mean hits "
              f"{float(best_hits.float().mean()):7.2f}")

    return best_z, best_hits


def cosine_rows(a, b):
    a = a / a.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    b = b / b.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return (a * b).sum(dim=-1)


def inversion_for(modality):
    sealed = json.loads(HELDOUT_RESULTS[modality].read_text())
    M, q, overlap = sealed["M"], sealed["q"], sealed["o"]
    tau = int(sealed["c_tau_carried_from_DEV"])
    subjects, enroll, probes = (
        face_heldout_data() if modality == "face" else voice_heldout_data()
    )
    key = selected_poly_key(modality)
    n, d = len(subjects), len(enroll[subjects[0]])
    k = 1 + math.ceil((d - G) / (G - overlap))
    print(f"  {modality}: n={n} d={d} k={k} M={M} q={q} tau={tau}")

    assert_diff_matches(modality, key, overlap, d)

    frozen = load_iom_tensor(modality, overlap, d)[:M, :q].to(DEVICE, torch.float32)
    recognition = {
        "polyiom": frozen,
        "iom_only": abl_randn((M, q, d), modality, "match|iom_only")[:M, :q].to(DEVICE),
        "randproj_iom": abl_randn((M, q, k), modality, "match|randproj")[:M, :q].to(DEVICE),
    }
    A = abl_randn((d, k), modality, "projmap").to(DEVICE) / math.sqrt(d)

    cross_generator = torch.Generator(device="cpu")
    cross_generator.manual_seed(seed_unlink(modality, 1))
    cross = {
        "polyiom": torch.randn((M, q, k), generator=cross_generator,
                               dtype=torch.float32).to(DEVICE),
        "iom_only": abl_randn((M, q, d), modality, "unlink|raw|1").to(DEVICE),
        "randproj_iom": abl_randn((M, q, k), modality, "unlink|prj|1").to(DEVICE),
    }

    truth = torch.as_tensor(
        np.stack([enroll[uid] for uid in subjects]),
        dtype=torch.float32, device=DEVICE)

    out = {}
    for arm in ARMS:
        print(f"    --- {arm}")
        targets = codes_of(arm, truth, key, overlap, recognition[arm], A)

        if arm == "polyiom":                       # assertion 2
            stored = torch.as_tensor(
                hash_arm([enroll[uid] for uid in subjects], arm, key, overlap,
                         recognition[arm], M, q, A).astype(np.int64),
                device=DEVICE)
            if not bool((stored == targets).all()):
                raise RuntimeError(
                    f"{modality}: attack targets do not equal the stored "
                    f"template; the attack would be aiming at the wrong thing."
                )
            print("    assertion 2 OK: attack targets equal the stored "
                  "templates, so a perfect solution exists in the search space")

        guess_generator = torch.Generator(device="cpu")
        guess_generator.manual_seed(H(S0, "INVERSION-GUESS", modality, arm))
        guess = torch.randn((n, d), generator=guess_generator, dtype=torch.float32)
        guess = (guess / guess.norm(dim=-1, keepdim=True)).to(DEVICE)
        guess_hits = (codes_of(arm, guess, key, overlap, recognition[arm], A)
                      == targets).sum(dim=-1)

        recovered, hits = invert_arm(
            arm, targets, key, overlap, recognition[arm], A, d,
            seed=H(S0, "INVERSION", modality, arm) % (2 ** 31))

        cross_true = codes_of(arm, truth, key, overlap, cross[arm], A)
        cross_rec = codes_of(arm, recovered, key, overlap, cross[arm], A)
        cross_hits = (cross_true == cross_rec).sum(dim=-1)
        cross_impostor = float(
            (cross_true[:, None, :] == cross_true[None, :, :]).sum(-1)
            [~torch.eye(n, dtype=torch.bool, device=DEVICE)].float().mean())

        cos_true = cosine_rows(recovered, truth)
        chance = cosine_rows(truth[:, None, :].expand(n, n, d).reshape(-1, d),
                             truth[None, :, :].expand(n, n, d).reshape(-1, d))
        chance = chance.reshape(n, n)[~torch.eye(n, dtype=torch.bool,
                                                 device=DEVICE)]

        out[arm] = {
            "hits": hits.cpu().numpy(),
            "guess_hits": guess_hits.cpu().numpy(),
            "cos": cos_true.cpu().numpy(),
            "cross_hits": cross_hits.cpu().numpy(),
            "cross_impostor_mean": cross_impostor,
            "cos_chance_mean": float(chance.mean()),
        }
        accepted = float((hits >= tau).float().mean())
        print(f"    {arm:<13} SAR {accepted * 100:6.2f}%   "
              f"mean hits {float(hits.float().mean()):6.2f}/{M}   "
              f"random guess {float(guess_hits.float().mean()):5.2f}/{M} "
              f"(chance {M / q:.1f})")

    return {"modality": modality, "subjects": subjects, "M": M, "tau": tau,
            "d": d, "k": k, "n": n, "arms": out, "chance": M / q}


def run_inversion(n_bootstrap=INV_N_BOOTSTRAP, confidence=INV_CONFIDENCE):
    output = DIR["runs"] / "inversion" / "inversion_result.json"
    if output.is_file():
        print("Resume: stored inversion result found; returning it unchanged.")
        return json.loads(output.read_text())

    result = {
        "schema": 1, "analysis": "full_knowledge_inversion",
        "adversary": "level 3: holds K*, R, the algorithm and the template",
        "attack": "annealed-softmax gradient descent on the unit sphere, "
                  "identical budget for every arm",
        "budget": {"restarts": INV_RESTARTS, "steps": INV_STEPS,
                   "lr": INV_LR, "temperature": [INV_TEMP0, INV_TEMP1]},
        "measures": "practical inversion resistance under this budget; "
                    "NOT information-theoretic irreversibility",
        "method_id": "identity_cluster_percentile_v1_paired",
        "n_bootstrap": int(n_bootstrap), "confidence": float(confidence),
        "master_seed": S0, "changes_any_seal": False, "modalities": {},
    }

    for modality in ("voice", "face"):
        print(f"\n{modality.upper()}")
        record = inversion_for(modality)
        n, tau, M = record["n"], record["tau"], record["M"]
        rng = np.random.default_rng(H(S0, "INVERSION-BOOTSTRAP", modality))

        accepted = {a: (record["arms"][a]["hits"] >= tau).astype(float)
                    for a in ARMS}
        draws = {a: [] for a in ARMS}
        contrasts = {a: [] for a in ARMS if a != "polyiom"}
        for _ in range(n_bootstrap):
            w = rng.multinomial(n, np.full(n, 1 / n)) / float(n)
            values = {a: float(w @ accepted[a]) for a in ARMS}
            for a in ARMS:
                draws[a].append(values[a])
            for a in contrasts:
                contrasts[a].append(values[a] - values["polyiom"])

        alpha = 1.0 - confidence

        def interval(v):
            lo, hi = np.quantile(v, [alpha / 2, 1 - alpha / 2])
            return [float(lo), float(hi)]

        result["modalities"][modality] = {
            "n_subjects": n, "M": M, "tau": tau, "d": record["d"],
            "k": record["k"],
            "arms": {a: {
                "SAR": float(accepted[a].mean()),
                "SAR_CI": interval(draws[a]),
                "mean_collisions": float(record["arms"][a]["hits"].mean()),
                "random_guess_collisions":
                    float(record["arms"][a]["guess_hits"].mean()),
                "chance_collisions": record["chance"],
                "cos_to_true": float(record["arms"][a]["cos"].mean()),
                "cos_chance": record["arms"][a]["cos_chance_mean"],
                "cross_key_collisions":
                    float(record["arms"][a]["cross_hits"].mean()),
                "cross_key_impostor_mean":
                    record["arms"][a]["cross_impostor_mean"],
            } for a in ARMS},
            "contrasts_vs_polyiom": {a: {
                "delta_SAR": float(accepted[a].mean()
                                   - accepted["polyiom"].mean()),
                "delta_SAR_CI": interval(contrasts[a]),
            } for a in contrasts},
        }

    atomic_json(output, result)
    print("\nWrote", output)
    return result


def inversion_report(result):
    for modality, data in result["modalities"].items():
        print(f"\n{modality.upper()}  n={data['n_subjects']}  M={data['M']}  "
              f"tau={data['tau']}")
        print(f"  {'arm':<14}{'SAR %':>22}{'hits/M':>10}{'guess':>8}"
              f"{'cos':>8}{'chance':>8}{'xkey':>7}{'xkey imp':>10}")
        for arm, v in data["arms"].items():
            print(f"  {arm:<14}{v['SAR'] * 100:7.2f} "
                  f"[{v['SAR_CI'][0] * 100:5.2f},{v['SAR_CI'][1] * 100:6.2f}]"
                  f"{v['mean_collisions']:10.1f}"
                  f"{v['random_guess_collisions']:8.1f}"
                  f"{v['cos_to_true']:8.3f}{v['cos_chance']:8.3f}"
                  f"{v['cross_key_collisions']:7.1f}"
                  f"{v['cross_key_impostor_mean']:10.1f}")
        chance = list(data["arms"].values())[0].get("chance_collisions")
        if chance:
            print(f"  a random unit vector lands on {chance:.1f} of "
                  f"{data['M']} buckets by chance; 'guess' should sit near it")
        print("  contrasts (arm minus proposed; interval excluding 0 is firm)")
        for arm, c in data["contrasts_vs_polyiom"].items():
            lo, hi = c["delta_SAR_CI"]
            firm = "FIRM" if (lo > 0 or hi < 0) else "not distinguishable"
            print(f"    {arm:<14} dSAR {c['delta_SAR'] * 100:+7.2f} %  "
                  f"[{lo * 100:+7.2f},{hi * 100:+7.2f}]  {firm}")
    print("\n  A POSITIVE dSAR means that arm is EASIER to invert than the")
    print("  proposed method, i.e. the polynomial helped. 'cos' against")
    print("  'chance' says whether the biometric itself was recovered;")
    print("  'xkey' against 'xkey imp' says whether the attack found the")
    print("  subject or merely a preimage of this one hash.")


print("Inversion evaluation ready.")


In [ ]:
%%writefile {RUNTIME}/revocation_only.py
# 5. The study's revocation test
"""Post-revocation acceptance for AttackAware PolyIoM v1.1.4.

Loaded after ablation_only.py and inversion_only.py. Answers the one
question the inversion run left open: does the attacker's reconstruction,
built from an OLD template, still get in after the subject re-enrols under
a FULLY re-keyed system?

The inversion run re-keyed the IoM projection R only. This re-keys
everything each arm has:

  polyiom       fresh polynomial key K** AND fresh projection R'
  randproj_iom  fresh random map A'      AND fresh projection R'
  iom_only      fresh projection R'      (it has nothing else)

The attacker never re-attacks. They hold one stolen template, reconstruct
once, and the subject then re-enrols. That is the real revocation
scenario.

Headline metric: **PRAR**, post-revocation acceptance rate - the fraction
of subjects whose old-template reconstruction is accepted at the sealed
threshold after full re-keying. Revocation works only if PRAR falls to the
impostor floor.

The first version of this used 3 fresh keys and bootstrapped over
identities. That was the wrong axis: with one catastrophic key out of
three, an identity bootstrap concentrates tightly around 1/3 and reports a
confidence interval for a question nobody asked. The dominant uncertainty
is over KEYS. This version draws 40 and bootstraps over both, and reports
the per-key distribution as well as the mean, because "fails for 1 key in
3" and "fails 34% of the time" are different claims.

K** is drawn to match K*'s structure by resampling its own coefficient and
exponent values, NOT by re-running Stage A's inversion gate. For this
question that is the right construction: we are asking whether z'
resembles z in the ways any polynomial of that family sees, not whether
K** is a good key. Three independent draws guard against a fluke.

Writes one file, runs/revocation/revocation_result.json, and no seal.
"""

REV_N_KEYS = 40
REV_N_BOOTSTRAP = 2000
REV_CONFIDENCE = 0.95


def fresh_poly_key(modality, key, index):
    """A new polynomial key matched to K*'s own value distribution."""
    C, E = key
    rng = np.random.default_rng(H(S0, "REVOCATION-KEY", modality, index))
    return (rng.choice(np.asarray(C).reshape(-1), size=np.asarray(C).shape,
                       replace=True).astype(np.float32),
            rng.choice(np.asarray(E).reshape(-1), size=np.asarray(E).shape,
                       replace=True).astype(np.int64))


def fresh_params(arm, modality, index, key, M, q, d, k):
    """Everything this arm would re-issue on revocation."""
    label = f"revocation|{index}"
    if arm == "polyiom":
        return (fresh_poly_key(modality, key, index),
                None,
                abl_randn((M, q, k), modality, f"{label}|R|poly").to(DEVICE))
    if arm == "randproj_iom":
        return (key,
                abl_randn((d, k), modality, f"{label}|A").to(DEVICE) / math.sqrt(d),
                abl_randn((M, q, k), modality, f"{label}|R|prj").to(DEVICE))
    if arm == "iom_only":
        return (key, None,
                abl_randn((M, q, d), modality, f"{label}|R|raw").to(DEVICE))
    raise ValueError(arm)


def revocation_for(modality):
    sealed = json.loads(HELDOUT_RESULTS[modality].read_text())
    M, q, overlap = sealed["M"], sealed["q"], sealed["o"]
    tau = int(sealed["c_tau_carried_from_DEV"])
    subjects, enroll, probes = (
        face_heldout_data() if modality == "face" else voice_heldout_data()
    )
    key = selected_poly_key(modality)
    n, d = len(subjects), len(enroll[subjects[0]])
    k = 1 + math.ceil((d - G) / (G - overlap))
    print(f"  {modality}: n={n} d={d} k={k} M={M} q={q} tau={tau}")
    assert_diff_matches(modality, key, overlap, d)

    frozen = load_iom_tensor(modality, overlap, d)[:M, :q].to(DEVICE, torch.float32)
    recognition = {
        "polyiom": frozen,
        "iom_only": abl_randn((M, q, d), modality, "match|iom_only").to(DEVICE),
        "randproj_iom": abl_randn((M, q, k), modality, "match|randproj").to(DEVICE),
    }
    A = abl_randn((d, k), modality, "projmap").to(DEVICE) / math.sqrt(d)
    truth = torch.as_tensor(np.stack([enroll[uid] for uid in subjects]),
                            dtype=torch.float32, device=DEVICE)

    out = {}
    for arm in ARMS:
        print(f"    --- {arm}")
        targets = codes_of(arm, truth, key, overlap, recognition[arm], A)
        # Same seeds as the inversion run, so z' is bit-identical to it.
        recovered, hits = invert_arm(
            arm, targets, key, overlap, recognition[arm], A, d,
            seed=H(S0, "INVERSION", modality, arm) % (2 ** 31))

        accepted_per_key, impostor_per_key, degenerate = [], [], []
        for index in range(REV_N_KEYS):
            new_key, new_A, new_R = fresh_params(
                arm, modality, index, key, M, q, d, k)
            eff_A = new_A if new_A is not None else A
            true_codes = codes_of(arm, truth, new_key, overlap, new_R, eff_A)
            off = ~torch.eye(n, dtype=torch.bool, device=DEVICE)
            floor = float((true_codes[:, None, :] == true_codes[None, :, :])
                          .sum(-1)[off].float().mean())
            # A resampled key can pair a large exponent with a small
            # coefficient so hard that every power underflows, the projection
            # collapses to zero, and argmax returns bucket 0 for everyone.
            # That would read as PRAR 100% for entirely the wrong reason.
            # Such keys are recorded and excluded, not silently averaged in:
            # how many there are is itself a finding about the key family.
            if floor > M / 4:
                degenerate.append(index)
                continue
            rec_codes = codes_of(arm, recovered, new_key, overlap, new_R, eff_A)
            agree = (true_codes == rec_codes).sum(dim=-1)
            accepted_per_key.append((agree >= tau).float().cpu().numpy())
            impostor_per_key.append(floor)

        if len(accepted_per_key) < 2:
            raise RuntimeError(
                f"{modality}/{arm}: only {len(accepted_per_key)} of "
                f"{REV_N_KEYS} fresh keys were usable; the key family is too "
                f"fragile for this test to say anything."
            )
        matrix = np.stack(accepted_per_key)          # (usable_keys, n)
        per_key = matrix.mean(axis=1)                # PRAR of each key
        print(f"      {len(matrix)}/{REV_N_KEYS} keys usable"
              f"{f' ({len(degenerate)} degenerate, excluded)' if degenerate else ''}"
              f"   per-key PRAR: median {np.median(per_key) * 100:6.2f}%"
              f"  min {per_key.min() * 100:6.2f}%  max {per_key.max() * 100:6.2f}%"
              f"   keys failing outright (>50%): "
              f"{float((per_key >= 0.5).mean()) * 100:5.1f}%")

        out[arm] = {
            "matrix": matrix,                          # (keys, identities)
            "per_key": per_key,
            "cos": cosine_rows(recovered, truth).cpu().numpy(),
            "hits": hits.cpu().numpy(),
            "impostor_floor": float(np.mean(impostor_per_key)),
            "n_degenerate": len(degenerate),
        }
        print(f"    {arm:<13} PRAR {matrix.mean() * 100:6.2f}%   "
              f"cos {out[arm]['cos'].mean():.3f}")

    return {"modality": modality, "M": M, "tau": tau, "d": d, "k": k,
            "n": n, "arms": out}


def run_revocation(n_bootstrap=REV_N_BOOTSTRAP, confidence=REV_CONFIDENCE):
    output = DIR["runs"] / "revocation" / "revocation_result.json"
    if output.is_file():
        print("Resume: stored revocation result found; returning it unchanged.")
        return json.loads(output.read_text())

    result = {
        "schema": 1, "analysis": "post_revocation_acceptance",
        "question": "does an old template's reconstruction still get in "
                    "after the subject re-enrols under a fully re-keyed system",
        "re_keys": {"polyiom": "fresh K** and fresh R",
                    "randproj_iom": "fresh A and fresh R",
                    "iom_only": "fresh R"},
        "n_fresh_keys": REV_N_KEYS,
        "fresh_key_construction": "resampled from K*'s own coefficient and "
                                  "exponent values; NOT passed through "
                                  "Stage A's inversion gate",
        "method_id": "identity_cluster_percentile_v1_paired",
        "n_bootstrap": int(n_bootstrap), "confidence": float(confidence),
        "master_seed": S0, "changes_any_seal": False, "modalities": {},
    }

    for modality in ("voice", "face"):
        print(f"\n{modality.upper()}")
        record = revocation_for(modality)
        n = record["n"]
        rng = np.random.default_rng(H(S0, "REVOCATION-BOOTSTRAP", modality))
        stats = {a: {"PRAR": [], "cos": []} for a in ARMS}
        contrasts = {a: {"delta_PRAR": [], "delta_cos": []}
                     for a in ARMS if a != "polyiom"}
        n_keys = min(len(record["arms"][a]["matrix"]) for a in ARMS)

        # Two-level: resample KEYS and IDENTITIES in every replicate, because
        # both vary and an identity-only bootstrap understates the spread.
        # Key index and identity weights are shared across arms, so a contrast
        # is paired on both.
        for _ in range(n_bootstrap):
            picks = rng.integers(0, n_keys, size=n_keys)
            w = rng.multinomial(n, np.full(n, 1 / n)) / float(n)
            values = {}
            for a in ARMS:
                arm_matrix = record["arms"][a]["matrix"][:n_keys]
                values[a] = (float(np.mean(arm_matrix[picks] @ w)),
                             float(w @ record["arms"][a]["cos"]))
                stats[a]["PRAR"].append(values[a][0])
                stats[a]["cos"].append(values[a][1])
            for a in contrasts:
                contrasts[a]["delta_PRAR"].append(values[a][0] - values["polyiom"][0])
                contrasts[a]["delta_cos"].append(values[a][1] - values["polyiom"][1])

        alpha = 1.0 - confidence

        def interval(v):
            lo, hi = np.quantile(v, [alpha / 2, 1 - alpha / 2])
            return [float(lo), float(hi)]

        result["modalities"][modality] = {
            "n_subjects": n, "M": record["M"], "tau": record["tau"],
            "d": record["d"], "k": record["k"],
            "bootstrap_axes": "keys and identities",
            "arms": {a: {
                "PRAR": float(record["arms"][a]["matrix"].mean()),
                "PRAR_CI": interval(stats[a]["PRAR"]),
                "per_key_PRAR": [float(x) for x in record["arms"][a]["per_key"]],
                "per_key_median": float(np.median(record["arms"][a]["per_key"])),
                "per_key_min": float(record["arms"][a]["per_key"].min()),
                "per_key_max": float(record["arms"][a]["per_key"].max()),
                "keys_failing_outright":
                    float((record["arms"][a]["per_key"] >= 0.5).mean()),
                "n_keys_usable": int(len(record["arms"][a]["matrix"])),
                "n_keys_degenerate": int(record["arms"][a]["n_degenerate"]),
                "cos_to_true": float(record["arms"][a]["cos"].mean()),
                "cos_CI": interval(stats[a]["cos"]),
                "impostor_floor_collisions": record["arms"][a]["impostor_floor"],
                "attack_collisions": float(record["arms"][a]["hits"].mean()),
            } for a in ARMS},
            "contrasts_vs_polyiom": {a: {
                "delta_PRAR": float(record["arms"][a]["matrix"].mean()
                                    - record["arms"]["polyiom"]["matrix"].mean()),
                "delta_PRAR_CI": interval(contrasts[a]["delta_PRAR"]),
                "delta_cos": float(record["arms"][a]["cos"].mean()
                                   - record["arms"]["polyiom"]["cos"].mean()),
                "delta_cos_CI": interval(contrasts[a]["delta_cos"]),
            } for a in contrasts},
        }

    atomic_json(output, result)
    print("\nWrote", output)
    return result


def revocation_report(result):
    for modality, data in result["modalities"].items():
        print(f"\n{modality.upper()}  n={data['n_subjects']}  M={data['M']}  "
              f"tau={data['tau']}")
        print(f"  {'arm':<14}{'PRAR %':>22}{'per-key med/min/max %':>26}"
              f"{'keys fail':>11}{'keys ok':>9}")
        for arm, v in data["arms"].items():
            print(f"  {arm:<14}"
                  f"{v['PRAR'] * 100:7.2f} [{v['PRAR_CI'][0] * 100:5.2f},"
                  f"{v['PRAR_CI'][1] * 100:6.2f}]"
                  f"{v['per_key_median'] * 100:9.2f}"
                  f"{v['per_key_min'] * 100:7.2f}"
                  f"{v['per_key_max'] * 100:7.2f}"
                  f"{v['keys_failing_outright'] * 100:10.1f}%"
                  f"{v['n_keys_usable']:6d}/{v['n_keys_usable'] + v['n_keys_degenerate']}")
        print(f"  {'':14}{'cosine to true embedding':>22}")
        for arm, v in data["arms"].items():
            print(f"  {arm:<14}{v['cos_to_true']:11.3f} "
                  f"[{v['cos_CI'][0]:6.3f},{v['cos_CI'][1]:6.3f}]")
        print("  contrasts (arm minus proposed; interval excluding 0 is firm)")
        for arm, c in data["contrasts_vs_polyiom"].items():
            for name, scale, unit in (("PRAR", 100, " %"), ("cos", 1, "")):
                value, (lo, hi) = c[f"delta_{name}"], c[f"delta_{name}_CI"]
                firm = "FIRM" if (lo > 0 or hi < 0) else "not distinguishable"
                print(f"    {arm:<14} d{name:<5}{value * scale:+8.3f}{unit}  "
                      f"[{lo * scale:+7.3f},{hi * scale:+7.3f}]  {firm}")
    print("\n  PRAR is the fraction of subjects whose OLD template's")
    print("  reconstruction is still accepted after full re-keying.")
    print("  Intervals resample BOTH keys and identities.")
    print("  'keys fail' is the share of fresh keys for which revocation")
    print("  fails outright (PRAR >= 50%) - the number to quote when the")
    print("  per-key spread is wide, because a mean over keys hides it.")
    print("  'keys ok' counts keys that stayed discriminating; degenerate")
    print("  ones are excluded, and a large exclusion count is itself a")
    print("  finding about how fragile the key family is.")


print("Revocation evaluation ready.")


In [ ]:
%%writefile {RUNTIME}/figures_only.py
# 6. The study's code for Figures A to D
"""Results figures for AttackAware PolyIoM v1.1.4.

Reads runs/scores/score_histograms.json and writes four figures, one per
ISO/IEC 24745 criterion the paper has to address. Read-only: touches no
seal, fits no threshold, computes no new score.

  A  det_preservation     the protected system's DET curves, and what
                          protection cost against the unprotected baseline
  B  score_separability   genuine and impostor, with tau* marked
  C  unlinkability        mated and non-mated, with D_link(s) overlaid
  D  revocability         genuine, impostor and pseudo-impostor together

Colour carries one meaning across all four figures
--------------------------------------------------
  blue    genuine comparisons, or the protected system
  grey    impostor comparisons, or non-mated pairs
  green   pseudo-impostor comparisons (a revoked key), or face
  orange  the unprotected baseline, and only ever that

Why figure A does not draw the baseline as a DET curve on voice
---------------------------------------------------------------
The unprotected voice baseline produces fewer than one genuine error: the
resolution of the evaluation split is 0.17% and of the external split
0.09%, and the measured EERs are 0.085% and 0.003%. A DET curve through
that region would sit off the bottom-left of any readable axis and would
imply a precision the trial count cannot support. The ratio it suggests
(22.7x held-out, 848x external) is an artifact of dividing by an
unresolved number.

The right panel therefore reports EER directly, with the baseline drawn as
a one-sided 95% upper bound wherever it is unresolved and as a point only
where it is genuinely measured. A reader can then see which contrasts the
data settle and which it does not.

Why figure C marks part of its own curve unreliable
---------------------------------------------------
The local unlinkability score D_link(s) is a ratio of two densities, and
the mated density is estimated from only one comparison per identity. In
score bins where the non-mated histogram is nearly empty the ratio is
driven by single observations and the curve runs to 1 for that reason
alone. Those bins are shaded rather than silently plotted.

Why the mated distribution appears twice
----------------------------------------
Figure C uses it as the unlinkability mated set and figure D uses it as
the revocability pseudo-impostor set. It is one measurement. Both captions
say so, and the paper must not count it as two independent results.
"""

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator

GEN = "#1c5cab"         # genuine, and the protected system
IMP = "#8a919c"         # impostor, and non-mated
PSE = "#158a5f"         # pseudo-impostor, and face
BASE = "#c94e1f"        # the unprotected baseline, and nothing else
INK = "#1a1a1a"
MUTE = "#6b7280"
GRID = "#d9dde3"
WARN = "#9aa1ab"
DERIV = INK          # a derived measure, never a data series

COL_W = 7.16            # double-column width, inches
DPI = 400

# Bootstrap intervals carried from the sealed held-out and external
# analyses. Not recomputed here: the histograms record distributions, not
# the identity-cluster resampling those intervals came from. Development
# is the selection split and was never given an interval.
CARRIED_CI = {
    ("voice", "evaluation"): (1.928, 1.092, 2.919),
    ("voice", "external"): (2.798, 1.992, 3.807),
    ("face", "evaluation"): (4.799, 1.945, 10.345),
}

ROWS = [("voice", "development"), ("voice", "evaluation"),
        ("voice", "external"), ("face", "development"),
        ("face", "evaluation")]
PANELS = [("voice", "evaluation"), ("face", "evaluation")]
LABEL = {("voice", "development"): "Voice, development (42)",
         ("voice", "evaluation"): "Voice, held-out (58)",
         ("voice", "external"): "Voice, external VCTK (110)",
         ("face", "development"): "Face, development (42)",
         ("face", "evaluation"): "Face, held-out (58)"}
TITLE = {("voice", "evaluation"): "Voice, 58 held-out speakers",
         ("face", "evaluation"): "Face, 58 held-out subjects",
         ("voice", "external"): "Voice, 110 external speakers",
         ("voice", "development"): "Voice, 42 development speakers",
         ("face", "development"): "Face, 42 development subjects"}

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8, "axes.labelsize": 8,
    "axes.titlesize": 8.5, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "axes.edgecolor": MUTE, "axes.linewidth": 0.6,
    "xtick.color": MUTE, "ytick.color": MUTE, "text.color": INK,
    "axes.labelcolor": INK, "axes.spines.top": False,
    "axes.spines.right": False, "figure.facecolor": "white",
    "savefig.facecolor": "white",
})


def _ndtri(p):
    from scipy.special import ndtri as f
    return f(np.clip(np.asarray(p, float), 1e-9, 1 - 1e-9))


def rates(genuine, impostor):
    """FMR and FNMR at every threshold, from two histograms.

    A comparison is accepted at a score of at least the threshold, so FMR
    is the upper tail of the impostor histogram and FNMR the lower tail of
    the genuine one.
    """
    g = np.asarray(genuine, float)
    i = np.asarray(impostor, float)
    fmr = np.r_[np.cumsum(i[::-1])[::-1], 0.0] / max(i.sum(), 1.0)
    fnmr = np.r_[0.0, np.cumsum(g)] / max(g.sum(), 1.0)
    return fmr, fnmr


def eer_from(genuine, impostor):
    fmr, fnmr = rates(genuine, impostor)
    d = fmr - fnmr
    exact = np.flatnonzero(d == 0.0)
    if len(exact):
        return float(fmr[exact[0]]), int(exact[0])
    cross = np.flatnonzero((d[:-1] > 0) & (d[1:] < 0))
    if len(cross):
        j = int(cross[0])
        w = d[j] / (d[j] - d[j + 1])
        return float(fmr[j] + w * (fmr[j + 1] - fmr[j])), j
    j = int(np.argmin(np.maximum(fmr, fnmr)))
    return float((fmr[j] + fnmr[j]) / 2.0), j


def baseline_bound(eer, n_genuine):
    """One-sided 95% upper bound on an unresolved baseline error rate.

    Returns (bound, resolved). An EER below one genuine error means the
    crossing sits where the genuine side has at most one mistake, so the
    point estimate carries no information and only a bound is honest.
    """
    from scipy.stats import beta
    floor = 1.0 / n_genuine
    k = int(np.floor(eer * n_genuine + 1e-9))
    bound = float(beta.ppf(0.95, k + 1, n_genuine - k))
    return bound, bool(eer >= floor)


def support(arrays, M, pad=0.10):
    """Score range actually occupied, so panels do not waste width.

    The distributions occupy a third of 0..M or less. Plotting the whole
    range leaves most of every panel blank and shrinks the part a reader
    needs to see. The axis label states that the axis is truncated.
    """
    lo, hi = M, 0
    for a in arrays:
        nz = np.flatnonzero(np.asarray(a, float) > 0)
        if len(nz):
            lo, hi = min(lo, int(nz[0])), max(hi, int(nz[-1]))
    if hi <= lo:
        return 0, M
    span = hi - lo
    return max(0, int(lo - pad * span)), min(M, int(hi + pad * span) + 1)


def _det_axes(ax, ticks=(0.1, 1, 5, 20, 50)):
    p = [t / 100 for t in ticks]
    pos = _ndtri(p)
    ax.set_xlim(pos[0], pos[-1])
    ax.set_ylim(pos[0], pos[-1])
    ax.xaxis.set_major_locator(FixedLocator(pos))
    ax.yaxis.set_major_locator(FixedLocator(pos))
    ax.set_xticklabels([f"{t:g}" for t in ticks])
    ax.set_yticklabels([f"{t:g}" for t in ticks])
    ax.grid(True, color=GRID, linewidth=0.5)
    ax.set_axisbelow(True)


def figure_a(data, outdir):
    fig, axes = plt.subplots(1, 2, figsize=(COL_W, 3.55),
                             gridspec_kw={"width_ratios": [1, 1.28]})

    # Left: the protected system's DET curves, which are measurable.
    ax = axes[0]
    curves = [(("voice", "evaluation"), GEN, "-", "Voice, held-out"),
              (("voice", "external"), GEN, (0, (4, 2)), "Voice, external"),
              (("face", "evaluation"), PSE, "-", "Face, held-out")]
    for (mod, part), colour, style, label in curves:
        parts = data["modalities"][mod]["partitions"]
        if part not in parts:
            continue
        h = parts[part]
        fmr, fnmr = rates(h["genuine"], h["impostor"])
        keep = (fmr > 0) & (fnmr > 0)
        eer, _ = eer_from(h["genuine"], h["impostor"])
        ax.plot(_ndtri(fmr[keep]), _ndtri(fnmr[keep]), color=colour,
                linewidth=1.5, linestyle=style, zorder=4,
                label=f"{label}  {eer * 100:.2f}%")
        p = _ndtri(eer)
        ax.plot([p], [p], "o", color=colour, markersize=3.6,
                markeredgecolor="white", markeredgewidth=0.6, zorder=5)
    # Stops short of the corner so the legend sits on clean ground.
    lim = _ndtri((0.001, 0.22))
    ax.plot(lim, lim, color=MUTE, linewidth=0.5, linestyle=(0, (4, 3)))
    _det_axes(ax)
    ax.set_xlabel("False match rate (%)")
    ax.set_ylabel("False non-match rate (%)")
    ax.set_title("Protected system, equal-error point marked", color=INK,
                 loc="left")
    ax.legend(loc="upper right", frameon=False, handlelength=1.8,
              borderpad=0.2, labelspacing=0.3)

    # Right: what protection cost, and what the data can resolve.
    ax = axes[1]
    ypos = np.arange(len(ROWS))[::-1]
    for y, (mod, part) in zip(ypos, ROWS):
        parts = data["modalities"][mod]["partitions"]
        if part not in parts:
            continue
        h = parts[part]
        prot, _ = eer_from(h["genuine"], h["impostor"])
        prot *= 100
        raw, _ = eer_from(h["unprotected"]["genuine"],
                          h["unprotected"]["impostor"])
        raw *= 100
        bound, resolved = baseline_bound(raw / 100, h["n_genuine"])
        bound *= 100

        # Two rows within a row: on face the protected and unprotected
        # points nearly coincide, and one would hide the other.
        yp, yb = y + 0.16, y - 0.16
        ci = CARRIED_CI.get((mod, part))
        if ci:
            ax.plot([ci[1], ci[2]], [yp, yp], color=GEN, linewidth=1.1,
                    solid_capstyle="butt", zorder=3)
            for e in (ci[1], ci[2]):
                ax.plot([e, e], [yp - 0.12, yp + 0.12], color=GEN,
                        linewidth=1.1, zorder=3)
        ax.plot([prot], [yp], "o", color=GEN, markersize=4.6,
                markeredgecolor="white", markeredgewidth=0.7, zorder=5)

        if resolved:
            ax.plot([raw], [yb], "s", color=BASE, markersize=4.2,
                    markeredgecolor="white", markeredgewidth=0.7, zorder=5)
            ax.plot([raw, bound], [yb, yb], color=BASE, linewidth=1.0,
                    alpha=0.55, zorder=3)
            ax.plot([bound, bound], [yb - 0.12, yb + 0.12], color=BASE,
                    linewidth=1.0, alpha=0.55, zorder=3)
            verdict = "not resolvable" if ci else "no interval"
        else:
            ax.annotate("", xy=(0.02, yb), xytext=(bound, yb),
                        arrowprops=dict(arrowstyle="-|>", color=BASE,
                                        linewidth=1.0, shrinkA=0, shrinkB=0),
                        zorder=4)
            ax.plot([bound], [yb], "|", color=BASE, markersize=7,
                    markeredgewidth=1.2, zorder=5)
            verdict = ("no interval" if not ci else
                       "firm" if ci[1] > bound else "not resolvable")
        ax.text(15.8, y, verdict, fontsize=6.4, va="center", ha="right",
                color=INK if verdict == "firm" else MUTE,
                style="normal" if verdict == "firm" else "italic")

    ax.set_yticks(ypos)
    ax.set_yticklabels([LABEL[r] for r in ROWS], fontsize=7)
    ax.set_xlim(0, 16)
    ax.set_ylim(-0.6, len(ROWS) - 0.4)
    ax.set_xlabel("Equal error rate (%)")
    ax.grid(True, axis="x", color=GRID, linewidth=0.5)
    ax.set_axisbelow(True)
    ax.set_title("Protection cost, and what the data settle", color=INK,
                 loc="left")
    handles = [
        plt.Line2D([], [], color=GEN, marker="o", linestyle="-",
                   markersize=4.6, linewidth=1.1,
                   label="Protected, 95% interval"),
        plt.Line2D([], [], color=BASE, marker="s", linestyle="-",
                   markersize=4.2, linewidth=1.0, alpha=0.7,
                   label="Unprotected, measured"),
        plt.Line2D([], [], color=BASE, marker="|", linestyle="none",
                   markersize=7, markeredgewidth=1.2,
                   label="Unprotected, 95% upper bound only"),
    ]
    # Below the axis: every in-panel corner is occupied by a row.
    ax.legend(handles=handles, loc="upper center",
              bbox_to_anchor=(0.5, -0.20), ncol=3, frameon=False,
              handlelength=1.5, borderpad=0.2, columnspacing=1.1,
              fontsize=6.0)
    fig.suptitle("Performance preservation against the unprotected "
                 "baseline", fontsize=9.5, x=0.005, ha="left", color=INK,
                 y=0.995)
    fig.tight_layout(rect=(0, 0.03, 1, 0.93))
    _save(fig, outdir, "figA_det_preservation")


def _dist_panel(ax, x, series, lo, hi, headroom=1.30):
    top = 0.0
    for label, raw, colour, style in series:
        v = np.asarray(raw, float)
        v = v / v.sum()
        top = max(top, v[lo:hi].max() if hi > lo else v.max())
        ax.fill_between(x, v, color=colour, alpha=0.42, linewidth=0, zorder=3)
        ax.plot(x, v, color=colour, linewidth=1.15, linestyle=style,
                zorder=4, label=label)
    ax.set_xlim(lo, hi)
    ax.set_ylim(0, top * headroom)
    return top


def figure_b(data, outdir):
    fig, axes = plt.subplots(1, 2, figsize=(COL_W, 2.9))
    for ax, (modality, partition) in zip(axes, PANELS):
        entry = data["modalities"][modality]
        h = entry["partitions"][partition]
        M, tau = entry["M"], entry["tau"]
        lo, hi = support([h["genuine"], h["impostor"]], M)
        x = np.arange(M + 1)
        top = _dist_panel(ax, x, [("Impostor", h["impostor"], IMP, "-"),
                                  ("Genuine", h["genuine"], GEN, "-")],
                          lo, hi)
        eer, _ = eer_from(h["genuine"], h["impostor"])
        ax.plot([tau, tau], [0, top * 0.95], color=INK, linewidth=0.9,
                zorder=5)
        ax.annotate(rf"$\tau^{{*}}={tau}$", xy=(tau, top * 0.055),
                    xytext=(4, 0), textcoords="offset points", fontsize=7,
                    color=INK, ha="left", va="bottom", zorder=6)
        ax.set_title(f"{TITLE[(modality, partition)]}   EER {eer * 100:.2f}%",
                     color=INK, loc="left")
        ax.set_xlabel(f"Collision score (axis truncated; $M$={M})")
        ax.legend(loc="upper right", frameon=False, handlelength=1.4,
                  borderpad=0.2)
    axes[0].set_ylabel("Share of comparisons")
    fig.suptitle("Where the sealed threshold sits in the score "
                 "distributions", fontsize=9.5, x=0.005, ha="left",
                 color=INK, y=0.995)
    fig.tight_layout(rect=(0, 0.02, 1, 0.92))
    _save(fig, outdir, "figB_score_separability")


def figure_c(data, outdir):
    fig, axes = plt.subplots(1, 2, figsize=(COL_W, 2.95))
    for ax, (modality, partition) in zip(axes, PANELS):
        entry = data["modalities"][modality]
        h = entry["partitions"][partition]
        M = entry["M"]
        lo, hi = support([h["mated"], h["nonmated"]], M)
        x = np.arange(M + 1)
        top = _dist_panel(ax, x, [("Non-mated", h["nonmated"], IMP, "-"),
                                  ("Mated", h["mated"], PSE, "-")], lo, hi,
                          headroom=1.52)
        ax.set_xlabel(f"Collision score (axis truncated; $M$={M})")

        # Bins where the non-mated side is nearly empty drive D_link(s) to
        # 1 on single observations. Shade them rather than imply a measure.
        non = np.asarray(h["nonmated"], float)
        thin = non < 5
        edges = np.flatnonzero(thin[:-1] != thin[1:])
        start = 0 if thin[0] else None
        for e in edges:
            if thin[e]:
                if start is not None:
                    ax.axvspan(start, e + 1, color=WARN, alpha=0.16,
                               linewidth=0, zorder=1)
                    start = None
            else:
                start = e + 1
        if start is not None:
            ax.axvspan(start, M, color=WARN, alpha=0.16, linewidth=0,
                       zorder=1)

        twin = ax.twinx()
        twin.plot(x, np.asarray(h["dlink_curve"], float), color=DERIV,
                  linewidth=1.25, zorder=5, label=r"$D_{\mathrm{link}}(s)$")
        twin.set_ylim(0, 1.32)
        twin.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
        twin.spines["top"].set_visible(False)
        twin.spines["right"].set_visible(True)
        twin.spines["right"].set_color(MUTE)
        twin.set_ylabel(r"$D_{\mathrm{link}}(s)$", color=DERIV)
        twin.tick_params(axis="y", colors=DERIV)

        ax.set_title(f"{TITLE[(modality, partition)]}   "
                     rf"$D^{{\mathrm{{sys}}}}_{{\leftrightarrow}}=$"
                     f"{h['dlink_global']:.3f}", color=INK, loc="left")
        hl, ll = ax.get_legend_handles_labels()
        h2, l2 = twin.get_legend_handles_labels()
        shade = plt.Rectangle((0, 0), 1, 1, color=WARN, alpha=0.16,
                              label="Fewer than 5 non-mated pairs")
        ax.legend(hl + h2 + [shade], ll + l2 + [shade.get_label()],
                  loc="upper right", frameon=False, handlelength=1.3,
                  borderpad=0.2, labelspacing=0.3, fontsize=6.4)
    axes[0].set_ylabel("Share of comparisons")
    fig.suptitle("Unlinkability: can two protected templates be tied to "
                 "one person?", fontsize=9.5, x=0.005, ha="left", color=INK,
                 y=0.995)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    _save(fig, outdir, "figC_unlinkability")


def figure_d(data, outdir):
    fig, axes = plt.subplots(1, 2, figsize=(COL_W, 3.05))
    for ax, (modality, partition) in zip(axes, PANELS):
        entry = data["modalities"][modality]
        h = entry["partitions"][partition]
        M, tau = entry["M"], entry["tau"]
        lo, hi = support([h["genuine"], h["impostor"], h["mated"]], M)
        x = np.arange(M + 1)
        top = _dist_panel(ax, x, [
            ("Impostor (other person)", h["impostor"], IMP, "-"),
            ("Pseudo-impostor (revoked key)", h["mated"], PSE, (0, (3, 1.6))),
            ("Genuine (current key)", h["genuine"], GEN, "-")], lo, hi,
            headroom=1.58)

        mated = np.asarray(h["mated"], float)
        prar = float(mated[tau:].sum() / mated.sum())
        ax.plot([tau, tau], [0, top * 0.95], color=INK, linewidth=0.9,
                zorder=5)
        ax.annotate(rf"$\tau^{{*}}={tau}$", xy=(tau, top * 0.055),
                    xytext=(4, 0), textcoords="offset points", fontsize=7,
                    color=INK, ha="left", va="bottom", zorder=6)
        ax.annotate(f"{prar * 100:.1f}% of revoked templates still\n"
                    f"accepted at $\\tau^{{*}}$ (PRAR, this key pair)",
                    xy=(0.98, 0.74), xycoords="axes fraction", fontsize=6.2,
                    color=INK, ha="right", va="top", zorder=6)
        ax.set_title(TITLE[(modality, partition)], color=INK, loc="left")
        ax.set_xlabel(f"Collision score (axis truncated; $M$={M})")
        ax.legend(loc="upper right", frameon=False, handlelength=1.5,
                  borderpad=0.2, labelspacing=0.3, fontsize=6.4)
    axes[0].set_ylabel("Share of comparisons")
    fig.suptitle("Revocability: does a template survive its own key being "
                 "revoked?", fontsize=9.5, x=0.005, ha="left", color=INK,
                 y=0.995)
    fig.tight_layout(rect=(0, 0.02, 1, 0.92))
    _save(fig, outdir, "figD_revocability")


def _save(fig, outdir, stem):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    for ext in ("pdf", "png"):
        fig.savefig(outdir / f"{stem}.{ext}", dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"  wrote {stem}.pdf and {stem}.png")


def make_figures(score_json=None, outdir=None):
    score_json = Path(score_json) if score_json else \
        DIR["runs"] / "scores" / "score_histograms.json"
    outdir = Path(outdir) if outdir else \
        score_json.parent.parent.parent / "figures" / "results"
    data = json.loads(score_json.read_text())
    if data.get("SYNTHETIC"):
        print("*** SYNTHETIC input: layout test only, not for the paper")
    print(f"Read {score_json}")
    for fn in (figure_a, figure_b, figure_c, figure_d):
        fn(data, outdir)
    print(f"\nFour figures in {outdir}")
    return outdir


print("Figure runtime ready. Run make_figures().")


In [ ]:
%%writefile {RUNTIME}/results_plots.py
# 7. Drawing code for Figures 3 to 8, S2 and P
"""Drawing code for the results figures of AttackAware PolyIoM v1.1.4.

One source for two callers:

  reproducibility/figures.py       draws from the stored result files
  ALL_FIGURES notebook (Colab)     draws from results it has just computed

Every function takes the data it draws as an argument and reads nothing
from disk, so the figures are only ever as good as their inputs. The data
layout is the one the study's own runtime writes (ablation, inversion and
revocation results), plus four small pieces described in draw_all().

Needs numpy and matplotlib only.
"""

from pathlib import Path

import matplotlib as mpl
mpl.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

# ------------------------------------------------------------------ style
SURFACE, INK, INK2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, DEEMPH, CRITICAL = "#e1e0d9", "#c3c2b7", "#c9c8c2", "#d03b3b"
VOICE, FACE = "#2a78d6", "#eb6834"
BLUES = LinearSegmentedColormap.from_list("blues", [
    "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95",
    "#0d366b"])

ARMS = ["polyiom", "iom_only", "randproj_iom"]
LABEL = {"polyiom": "PolyIoM (ours)", "iom_only": "IoM only",
         "randproj_iom": "Linear map + IoM"}
HUE = {"polyiom": VOICE, "iom_only": FACE, "randproj_iom": "#1baf7a"}
SHAPE = {"polyiom": "o", "iom_only": "s", "randproj_iom": "^"}

STYLE = {
    "font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans"],
    "font.size": 8.5, "axes.titlesize": 8.6, "axes.labelsize": 8.2,
    "xtick.labelsize": 7.6, "ytick.labelsize": 7.6, "legend.fontsize": 7.6,
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "text.color": INK,
    "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.6, "grid.color": GRID,
    "grid.linewidth": 0.6, "grid.linestyle": "-", "legend.frameon": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
}

def _check(cond, what):
    if not cond:
        raise ValueError(f"figure input check failed: {what}")


def close(a, b, tol):
    return abs(a - b) <= tol


def eligible(rows, mod, floor):
    e, t = floor[mod]
    return [r for r in rows if r["EER"] <= e and r["TAR_DEV"] >= t]


def save(fig, out, stem):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    # No timestamps in the files, so a rerun reproduces them byte for byte.
    fig.savefig(out / f"{stem}.pdf", bbox_inches="tight",
                metadata={"CreationDate": None, "ModDate": None})
    fig.savefig(out / f"{stem}.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  wrote {stem}.png and .pdf")
    return stem

def style(ax, grid_axis="both"):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(True, axis=grid_axis)
    ax.tick_params(length=3, pad=3)




def figure_selection(D, out):
    fig, (axa, axb) = plt.subplots(1, 2, figsize=(7.4, 3.2))
    style(axa)
    for mod, col in (("face", FACE), ("voice", VOICE)):
        rows = D["sweep"][mod]
        axa.scatter([r["EER"] * 100 for r in rows], [r["TAR_DEV"] * 100 for r in rows],
                    s=26, c=col, alpha=0.75, linewidths=0.7, edgecolors=SURFACE,
                    zorder=3, label=f"{mod} (80 settings)")
        e, t = D["floor"][mod]
        axa.add_patch(Rectangle((0, t * 100), e * 100, 100 - t * 100, facecolor=col,
                                alpha=0.07, edgecolor=col, linewidth=0.7, zorder=1))
    axa.annotate("voice floor", xy=(1.18, 97.8), fontsize=7.2, color=INK2, va="center")
    axa.annotate("face floor", xy=(3.18, 94.8), fontsize=7.2, color=INK2, va="center")
    for mod, col, xytext in (("voice", VOICE, (0.18, 86.5)), ("face", FACE, (4.30, 92.5))):
        s = D["selected"][mod]
        axa.scatter([s["EER"] * 100], [s["TAR_DEV"] * 100], s=124, c=col,
                    linewidths=1.8, edgecolors=SURFACE, zorder=5)
        axa.annotate("selected", xy=(s["EER"] * 100, s["TAR_DEV"] * 100), xytext=xytext,
                     fontsize=7.4, va="center",
                     arrowprops=dict(arrowstyle="-", color=MUTED, linewidth=0.7,
                                     shrinkA=3, shrinkB=7))
    axa.set_xlabel("EER (%), development")
    axa.set_ylabel("TAR (%), development")
    axa.set_xlim(0, 7.2)
    axa.set_ylim(35, 103)
    axa.set_title("(a)  No face setting meets the voice floor", loc="left", pad=7)
    axa.legend(loc="lower right", handletextpad=0.4, borderpad=0.2)

    style(axb)
    for mod, col in (("face", FACE), ("voice", VOICE)):
        el = eligible(D["sweep"][mod], mod, D["floor"])
        ine = [r for r in D["sweep"][mod] if r not in el]
        axb.scatter([r["EER"] * 100 for r in ine], [r["Dsys_DEV"] for r in ine], s=20,
                    c=DEEMPH, alpha=0.85, linewidths=0.6, edgecolors=SURFACE, zorder=2)
        axb.scatter([r["EER"] * 100 for r in el], [r["Dsys_DEV"] for r in el], s=26,
                    c=col, alpha=0.85, linewidths=0.7, edgecolors=SURFACE, zorder=3,
                    label=f"{mod}, meets floor ({len(el)})")
        s = D["selected"][mod]
        axb.scatter([s["EER"] * 100], [s["Dsys_DEV"]], s=124, c=col, linewidths=1.8,
                    edgecolors=SURFACE, zorder=5)
    axb.scatter([], [], s=20, c=DEEMPH, linewidths=0.6, edgecolors=SURFACE,
                label="below floor")
    axb.annotate("large mark = selected", xy=(7.05, 0.012), fontsize=7.0, color=MUTED,
                 ha="right", va="bottom")
    axb.set_xlabel("EER (%), development")
    axb.set_ylabel("$D_{sys}$, development  (lower is better)")
    axb.set_xlim(0, 7.2)
    axb.set_ylim(0, 0.41)
    axb.set_title("(b)  Lowest $D_{sys}$ that clears each floor", loc="left", pad=7)
    axb.legend(loc="upper right", handletextpad=0.4, borderpad=0.2)
    fig.tight_layout(pad=1.1)
    return save(fig, out, "FigureS2_OperatingPointSelection")


# ============================================== Figure 3: the design space
def figure_design_space(D, out, MS=(32, 64, 128, 256), QS=(4, 8, 16, 32), OS=(0, 1, 2, 3, 4)):
    vals = [r["Dsys_DEV"] for m in D["sweep"] for r in D["sweep"][m]]
    norm = Normalize(vmin=min(vals), vmax=max(vals))
    fig, axes = plt.subplots(2, 5, figsize=(7.5, 3.5),
                             gridspec_kw={"wspace": 0.16, "hspace": 0.13})
    for row, mod in enumerate(("face", "voice")):
        for col, o in enumerate(OS):
            ax = axes[row, col]
            g = np.full((len(MS), len(QS)), np.nan)
            for r in D["sweep"][mod]:
                if r["o"] == o:
                    g[MS.index(r["M"]), QS.index(r["q"])] = r["Dsys_DEV"]
            ax.imshow(g, cmap=BLUES, norm=norm, origin="upper", aspect="auto")
            for i in range(len(MS)):
                for j in range(len(QS)):
                    v = g[i, j]
                    ax.text(j, i, f"{v:.2f}".lstrip("0"), ha="center", va="center",
                            fontsize=5.9, color=SURFACE if norm(v) > 0.55 else INK2)
            s = D["selected"][mod]
            if s["o"] == o:
                ax.add_patch(plt.Rectangle((QS.index(s["q"]) - 0.5, MS.index(s["M"]) - 0.5),
                                           1, 1, fill=False, linewidth=1.9, zorder=5,
                                           edgecolor=FACE if mod == "face" else "#0d366b"))
            ax.set_xticks(range(len(QS)))
            ax.set_yticks(range(len(MS)))
            ax.set_xticklabels(QS if row == 1 else [])
            ax.set_yticklabels(MS if col == 0 else [])
            ax.tick_params(length=0, pad=2)
            for sp in ax.spines.values():
                sp.set_visible(False)
            if row == 0:
                ax.set_title(f"$o$ = {o}", pad=4, color=INK2)
            if col == 0:
                ax.set_ylabel(f"{mod}\n\n$M$", color=INK, labelpad=2)
            if row == 1:
                ax.set_xlabel("$q$")
    cb = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=BLUES), ax=axes,
                      fraction=0.022, pad=0.015)
    cb.set_label("$D_{sys}$, development  (lower is better)", color=INK2, fontsize=8)
    cb.outline.set_visible(False)
    cb.ax.tick_params(length=2, labelsize=7.4, color=AXIS)
    fig.suptitle("Unlinkability across the 80-setting design space; the ringed cell "
                 "is the sealed setting", fontsize=8.8, x=0.008, ha="left", y=0.985)
    return save(fig, out, "Figure3_DesignSpace")


# ================================================== Figure 4: the frontier
def pareto(rows):
    out, best = [], float("inf")
    for r in sorted(rows, key=lambda r: (r["EER"], r["Dsys_DEV"])):
        if r["Dsys_DEV"] < best:
            out.append(r)
            best = r["Dsys_DEV"]
    return out


def figure_frontier(D, out):
    fig, ax = plt.subplots(figsize=(4.9, 3.6))
    style(ax)
    for mod, col in (("face", FACE), ("voice", VOICE)):
        ok = eligible(D["sweep"][mod], mod, D["floor"])
        no = [r for r in D["sweep"][mod] if r not in ok]
        ax.scatter([r["EER"] * 100 for r in no], [r["Dsys_DEV"] for r in no], s=17,
                   c=DEEMPH, linewidths=0.6, edgecolors=SURFACE, zorder=2)
        ax.scatter([r["EER"] * 100 for r in ok], [r["Dsys_DEV"] for r in ok], s=24,
                   c=col, alpha=0.9, linewidths=0.7, edgecolors=SURFACE, zorder=3,
                   label=f"{mod}, meets floor ({len(ok)})")
        pf = pareto(D["sweep"][mod])
        ax.step([r["EER"] * 100 for r in pf], [r["Dsys_DEV"] for r in pf], where="post",
                color=col, linewidth=1.3, alpha=0.9, zorder=4)
        s = D["selected"][mod]
        ax.scatter([s["EER"] * 100], [s["Dsys_DEV"]], s=120, c=col, linewidths=1.8,
                   edgecolors=SURFACE, zorder=6)
    ax.scatter([], [], s=17, c=DEEMPH, linewidths=0.6, edgecolors=SURFACE,
               label="below floor")
    ax.plot([], [], color=MUTED, linewidth=1.3, label="frontier")
    ax.annotate("large mark = sealed setting", xy=(7.05, 0.015), fontsize=7.0,
                color=MUTED, ha="right", va="bottom")
    ax.set_xlabel("EER (%), development")
    ax.set_ylabel("$D_{sys}$, development  (lower is better)")
    ax.set_xlim(0, 7.2)
    ax.set_ylim(0, 0.40)
    ax.set_title("Recognition and unlinkability trade off across the design space",
                 loc="left", pad=7, fontsize=8.6)
    ax.legend(loc="upper right", handletextpad=0.5, borderpad=0.2)
    fig.tight_layout(pad=0.9)
    return save(fig, out, "Figure4_Frontier")


# ============================================ Figure 5: generalisation
def figure_generalisation(D, out):
    rows = [("face", "development (42)", None), ("face", "held-out (58)", "face/evaluation"),
            ("voice", "development (42)", None), ("voice", "held-out (58)", "voice/evaluation"),
            ("voice", "external, VCTK (110)", "voice/external")]
    dev = {m: {"EER": D["selected"][m]["EER"] * 100, "TAR": D["selected"][m]["TAR_DEV"] * 100,
               "FMR": D["selected"][m]["FMR_DEV"] * 100, "Dsys": D["selected"][m]["Dsys_DEV"]}
           for m in ("face", "voice")}
    panels = [("EER", "EER (%)", [(1.0, "1%"), (3.0, "3%")], (0, 11)),
              ("TAR", "TAR (%)", [], (65, 100)),
              ("FMR", "FMR (%)", [(0.1, "0.1%")], (0, 0.68)),
              ("Dsys", "$D_{sys}$", [], (0, 0.30))]
    fig, axes = plt.subplots(1, 4, figsize=(7.6, 3.05))
    ypos = list(range(len(rows)))[::-1]
    for ax, (key, xlabel, refs, xlim) in zip(axes, panels):
        style(ax, "x")
        for ref, lab in refs:
            ax.axvline(ref, color=MUTED, linewidth=0.7, zorder=1)
            ax.annotate(lab, xy=(ref, len(rows) - 0.38), fontsize=6.8, color=MUTED,
                        ha="center", va="bottom")
        for y, (mod, _, part) in zip(ypos, rows):
            col = VOICE if mod == "voice" else FACE
            if part:
                est, lo, hi = D["intervals"][part][key]
                ax.plot([lo, hi], [y, y], color=col, linewidth=1.8,
                        solid_capstyle="butt", zorder=3)
                for cap in (lo, hi):
                    ax.plot([cap, cap], [y - 0.16, y + 0.16], color=col, linewidth=1.8,
                            zorder=3)
                ax.scatter([est], [y], s=46, c=col, linewidths=1.5, edgecolors=SURFACE,
                           zorder=5)
            else:
                ax.scatter([dev[mod][key]], [y], s=34, facecolors=SURFACE,
                           linewidths=1.3, edgecolors=col, zorder=5)
        ax.set_yticks(ypos)
        ax.set_ylim(-0.6, len(rows) - 0.15)
        ax.set_xlim(*xlim)
        ax.set_xlabel(xlabel)
        ax.set_yticklabels([f"{m} · {lab}" for m, lab, _ in rows] if ax is axes[0] else [],
                           fontsize=7.4, color=INK2)
    handles = [Line2D([], [], color=FACE, linewidth=1.8, marker="o", markersize=5,
                      markeredgecolor=SURFACE, label="face"),
               Line2D([], [], color=VOICE, linewidth=1.8, marker="o", markersize=5,
                      markeredgecolor=SURFACE, label="voice"),
               Line2D([], [], color=MUTED, linewidth=0, marker="o", markersize=5,
                      markerfacecolor=SURFACE, markeredgecolor=MUTED,
                      label="development (no interval)")]
    fig.legend(handles=handles, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.005),
               handletextpad=0.4, columnspacing=1.8)
    fig.suptitle("Point estimates with 95% identity-cluster bootstrap intervals "
                 "(2,000 replicates)", fontsize=8.8, x=0.008, ha="left", y=0.99)
    fig.tight_layout(pad=1.0, rect=(0, 0.075, 1, 0.935))
    return save(fig, out, "Figure5_Generalisation")


# ============================================ Figure P: protection cost
def figure_protection_cost(D, out):
    order = [("voice", "development"), ("voice", "evaluation"), ("voice", "external"),
             ("face", "development"), ("face", "evaluation")]
    name = {"development": "development (42)", "evaluation": "held-out (58)",
            "external": "external, VCTK (110)"}
    parts = {(p["modality"], p["partition"]): p for p in D["baseline"]}
    fig, ax = plt.subplots(figsize=(7.0, 3.0))
    style(ax, "x")
    ypos = list(range(len(order)))[::-1]
    for y, key in zip(ypos, order):
        p = parts[key]
        ci = D["intervals"].get(f"{key[0]}/{key[1]}", {}).get("EER")
        yp, yb = y + 0.16, y - 0.16
        if ci:
            ax.plot([ci[1], ci[2]], [yp, yp], color=VOICE, linewidth=1.3, zorder=3)
            for e in ci[1:]:
                ax.plot([e, e], [yp - 0.1, yp + 0.1], color=VOICE, linewidth=1.3, zorder=3)
        ax.plot([p["protected_eer"]], [yp], "o", color=VOICE, markersize=5,
                markeredgecolor=SURFACE, zorder=5)
        bound = p["unprotected_upper_bound_95"]
        if p["baseline_resolved"]:
            ax.plot([p["unprotected_eer"]], [yb], "s", color=FACE, markersize=4.6,
                    markeredgecolor=SURFACE, zorder=5)
            ax.plot([p["unprotected_eer"], bound], [yb, yb], color=FACE, linewidth=1.0,
                    alpha=0.55, zorder=3)
            ax.plot([bound, bound], [yb - 0.1, yb + 0.1], color=FACE, linewidth=1.0,
                    alpha=0.55, zorder=3)
        else:
            ax.annotate("", xy=(0.02, yb), xytext=(bound, yb), zorder=4,
                        arrowprops=dict(arrowstyle="-|>", color=FACE, linewidth=1.0,
                                        shrinkA=0, shrinkB=0))
            ax.plot([bound], [yb], "|", color=FACE, markersize=7, markeredgewidth=1.2,
                    zorder=5)
        if not ci:
            verdict = "development: no interval"
        elif ci[1] > bound:
            verdict = f"+{p['cost_pp']:.2f} pp, firm"
        else:
            verdict = f"+{p['cost_pp']:.2f} pp, not resolvable"
        ax.text(16.4, y, verdict, fontsize=6.8, va="center", ha="right",
                color=INK if verdict.endswith("firm") else MUTED)
    ax.set_yticks(ypos)
    ax.set_yticklabels([f"{m} · {name[p]}" for m, p in order], fontsize=7.4, color=INK2)
    ax.set_xlim(0, 16.5)
    ax.set_xticks([0, 2, 4, 6, 8, 10, 12])
    ax.set_ylim(-0.6, len(order) - 0.4)
    ax.set_xlabel("Equal error rate (%)")
    handles = [Line2D([], [], color=VOICE, marker="o", markersize=5, linewidth=1.3,
                      label="protected, 95% interval"),
               Line2D([], [], color=FACE, marker="s", markersize=4.6, linewidth=1.0,
                      alpha=0.7, label="unprotected, measured"),
               Line2D([], [], color=FACE, marker="|", linestyle="none", markersize=7,
                      markeredgewidth=1.2, label="unprotected, 95% upper bound only")]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.45, -0.2), ncol=3,
              handlelength=1.5, columnspacing=1.2, fontsize=6.8)
    ax.set_title("What protection costs against the unprotected system",
                 loc="left", pad=7)
    fig.tight_layout(pad=0.9)
    return save(fig, out, "FigureP_ProtectionCost")


# =============================================== Figure 6: the ablation
def dot_interval(ax, values, xmax, pct=False):
    for i, arm in enumerate(ARMS):
        point, lo, hi = values[arm]
        y = len(ARMS) - 1 - i
        ax.plot([lo, hi], [y, y], color=HUE[arm], lw=2.0, solid_capstyle="round", zorder=3)
        ax.plot([point], [y], marker=SHAPE[arm], ms=7.5, color=HUE[arm], mec=SURFACE,
                mew=1.4, zorder=4)
        ax.annotate(f"{point:.2f}%" if pct else f"{point:.3f}", (hi, y), xytext=(5, 0),
                    textcoords="offset points", fontsize=7.2, color=INK2, va="center")
    ax.set_yticks(range(len(ARMS)))
    ax.set_yticklabels([LABEL[a] for a in reversed(ARMS)])
    ax.set_ylim(-0.6, len(ARMS) - 0.4)
    ax.grid(True, axis="x")
    ax.set_axisbelow(True)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.tick_params(length=0, pad=3)
    ax.set_xlim(0, xmax)


def figure_ablation(D, out):
    fig, axes = plt.subplots(2, 2, figsize=(7.2, 3.6),
                             gridspec_kw={"wspace": 0.78, "hspace": 0.70})
    for r, mod in enumerate(("voice", "face")):
        arms = D["ablation"][mod]["arms"]
        eer = {a: (arms[a]["EER"] * 100, arms[a]["EER_CI"][0] * 100,
                   arms[a]["EER_CI"][1] * 100) for a in ARMS}
        dsys = {a: (arms[a]["Dsys"], *arms[a]["Dsys_CI"]) for a in ARMS}
        dot_interval(axes[r, 0], eer, xmax=11.5 if mod == "face" else 3.6, pct=True)
        dot_interval(axes[r, 1], dsys, xmax=0.33)
        axes[r, 0].set_title(f"{mod} — equal error rate (%)", loc="left", pad=6)
        axes[r, 1].set_title(f"{mod} — $D_{{sys}}$", loc="left", pad=6)
        for c in (0, 1):
            axes[r, c].set_xlabel("lower is better")
    v = {a: D["ablation"]["voice"]["arms"][a]["EER"] * 100 for a in ARMS}
    ax = axes[0, 0]
    for x0, x1, yy, col, txt in (
            (v["iom_only"], v["randproj_iom"], -0.50, MUTED,
             f"+{v['randproj_iom'] - v['iom_only']:.2f} pp  from the size reduction"),
            (v["randproj_iom"], v["polyiom"], -1.05, HUE["polyiom"],
             f"+{v['polyiom'] - v['randproj_iom']:.2f} pp  from the polynomial itself")):
        ax.annotate("", xy=(x0, yy), xytext=(x1, yy),
                    arrowprops=dict(arrowstyle="|-|,widthA=0.22,widthB=0.22", color=col,
                                    lw=0.9))
        ax.annotate(txt, (x0, yy - 0.13), ha="left", va="top", fontsize=6.4, color=col)
    ax.set_ylim(-1.80, len(ARMS) - 0.4)
    fig.suptitle("What each stage costs: error rate and linkability of the three arms",
                 fontsize=9, x=0.008, ha="left", y=0.995)
    return save(fig, out, "Figure6_Ablation")


# ============================================ Figure 7: revocation per key
def figure_revocation(D, out):
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.45), gridspec_kw={"wspace": 0.30})
    rng = np.random.default_rng(D["revocation"]["master_seed"])
    n_keys = D["revocation"]["n_fresh_keys"]
    for c, mod in enumerate(("voice", "face")):
        m = D["revocation"]["modalities"][mod]
        ax = axes[c]
        counts = []
        for i, arm in enumerate(ARMS):
            vals = np.asarray(m["arms"][arm]["per_key_PRAR"], float) * 100
            _check(vals.size == n_keys, f"{mod}/{arm}: {vals.size} keys, expected {n_keys}")
            y = len(ARMS) - 1 - i
            ax.scatter(vals, y + rng.uniform(-0.17, 0.17, vals.size), s=15,
                       marker=SHAPE[arm], facecolor=HUE[arm], edgecolor=SURFACE,
                       linewidth=0.5, alpha=0.85, zorder=3)
            counts.append(int((vals >= 50).sum()))
        ax.axvline(50, color=CRITICAL, lw=0.9, ls=(0, (3, 2)), zorder=2)
        ax.annotate("half or more exposed", (50, len(ARMS) - 1.5), fontsize=6.4,
                    color=CRITICAL, ha="center", va="center", rotation=90,
                    bbox=dict(facecolor=SURFACE, edgecolor="none", pad=1.5))
        ax.set_yticks(range(len(ARMS)))
        ax.set_yticklabels([LABEL[a] for a in reversed(ARMS)] if c == 0 else [])
        ax.set_ylim(-0.55, len(ARMS) - 0.45)
        ax.set_xlim(-6, 106)
        ax.set_xticks([0, 25, 50, 75, 100])
        ax.grid(True, axis="x")
        ax.set_axisbelow(True)
        for s in ("top", "right", "left"):
            ax.spines[s].set_visible(False)
        ax.tick_params(length=0, pad=3)
        ax.set_title(rf"{mod}  (M = {m['M']}, $\tau^*$ = {m['tau']})", loc="left", pad=6)
        ax.set_xlabel("acceptance rate after re-keying (%)")
        right = ax.twinx()
        right.set_ylim(ax.get_ylim())
        right.set_yticks(range(len(ARMS)))
        right.set_yticklabels([f"{n} of {n_keys}" for n in reversed(counts)])
        right.tick_params(length=0, pad=4, labelsize=6.8)
        for s in ("top", "right", "left", "bottom"):
            right.spines[s].set_visible(False)
        for tick, n in zip(right.get_yticklabels(), reversed(counts)):
            tick.set_color(CRITICAL if n else MUTED)
    fig.suptitle("Each dot is one fresh key. Most PolyIoM keys revoke completely; a "
                 "minority do not", fontsize=9, x=0.008, ha="left", y=1.02)
    return save(fig, out, "Figure7_RevocationPerKey")


# ================================================ Figure 8: the inversion
def figure_inversion(D, out):
    fig, axes = plt.subplots(2, 2, figsize=(7.2, 3.6),
                             gridspec_kw={"wspace": 0.78, "hspace": 0.70})
    for r, mod in enumerate(("voice", "face")):
        inv = D["inversion"][mod]
        M, tau = inv["M"], inv["tau"]
        chance = inv["arms"]["polyiom"]["chance_collisions"]
        cos_chance = inv["arms"]["polyiom"]["cos_chance"]
        ax = axes[r, 0]
        for i, arm in enumerate(ARMS):
            v = inv["arms"][arm]["mean_collisions"]
            y = len(ARMS) - 1 - i
            ax.plot([0, v], [y, y], color=HUE[arm], lw=2.0, solid_capstyle="butt", zorder=3)
            ax.plot([v], [y], marker=SHAPE[arm], ms=7.5, color=HUE[arm], mec=SURFACE,
                    mew=1.4, zorder=4)
            ax.annotate(f"{v:.1f}", (v, y), xytext=(5, 0), textcoords="offset points",
                        fontsize=7.2, color=INK2, va="center")
        ax.axvline(tau, color=CRITICAL, lw=0.9, ls=(0, (3, 2)), zorder=2)
        ax.annotate(rf"$\tau^*$ = {tau}", (tau + 2, -0.50), fontsize=6.6, color=CRITICAL,
                    ha="left", va="center",
                    bbox=dict(facecolor=SURFACE, edgecolor="none", pad=1.2))
        ax.axvline(chance, color=MUTED, lw=0.8, zorder=2)
        ax.set_yticks(range(len(ARMS)))
        ax.set_yticklabels([LABEL[a] for a in reversed(ARMS)])
        ax.set_ylim(-0.75, len(ARMS) - 0.35)
        ax.set_xlim(0, M * 1.14)
        ax.grid(True, axis="x")
        ax.set_axisbelow(True)
        for s in ("top", "right", "left"):
            ax.spines[s].set_visible(False)
        ax.tick_params(length=0, pad=3)
        ax.set_title(f"{mod} — matching positions reached (of {M})", loc="left", pad=6)
        ax.set_xlabel(f"further right = more complete attack (chance {chance:.0f})")

        # The cosine intervals are stored with the revocation run, which
        # re-ran the same attack; the point estimates must agree.
        cos = {}
        for arm in ARMS:
            ra = D["revocation"]["modalities"][mod]["arms"][arm]
            _check(close(ra["cos_to_true"], inv["arms"][arm]["cos_to_true"], 1e-9),
                  f"{mod}/{arm}: inversion and revocation cosines differ")
            cos[arm] = (ra["cos_to_true"], *ra["cos_CI"])
        ax = axes[r, 1]
        dot_interval(ax, cos, xmax=1.06)
        ax.axvline(cos_chance, color=MUTED, lw=0.8, zorder=2)
        ax.annotate(f"chance {cos_chance:.3f}", (cos_chance, -0.50), fontsize=6.6,
                    color=MUTED, ha="left", va="center",
                    bbox=dict(facecolor=SURFACE, edgecolor="none", pad=1.2))
        ax.set_ylim(-0.75, len(ARMS) - 0.35)
        ax.set_title(f"{mod} — cosine to the true embedding", loc="left", pad=6)
        ax.set_xlabel("higher = more of the biometric recovered")
    fig.suptitle("The attack clears the threshold in every arm; only the recovered "
                 "embedding separates them", fontsize=9, x=0.008, ha="left", y=0.995)
    return save(fig, out, "Figure8_Inversion")



FIGURES = (figure_selection, figure_design_space, figure_frontier,
           figure_generalisation, figure_protection_cost, figure_ablation,
           figure_revocation, figure_inversion)


def draw_all(D, out):
    """Draw the eight results figures into `out`; return their file stems.

    D["sweep"]       {modality: [row, ...]}, 80 rows each, with keys
                     M, q, o, EER, TAR_DEV, FMR_DEV, Dsys_DEV (fractions)
    D["floor"]       {modality: (max EER, min TAR)} of the selection rule
    D["selected"]    {modality: the sweep row that was sealed}
    D["intervals"]   {"voice/evaluation" | "face/evaluation" |
                      "voice/external": {"EER" | "TAR" | "FMR": (point,
                      low, high) in percent, "Dsys": (point, low, high)}}
    D["baseline"]    [{modality, partition, protected_eer, unprotected_eer,
                      cost_pp, baseline_resolved,
                      unprotected_upper_bound_95}, ...] in percent
    D["ablation"]    ablation_result["modalities"]
    D["revocation"]  revocation_result (the whole record)
    D["inversion"]   inversion_result["modalities"]
    """
    stems = []
    for draw in FIGURES:
        # Start from matplotlib's defaults every time, so a style set by
        # other plotting code in the same session cannot leak in.
        with mpl.rc_context():
            mpl.rcdefaults()
            mpl.rcParams.update(STYLE)
            stems.append(draw(D, out))
    return stems


In [ ]:
%%writefile {RUNTIME}/allfigures_only.py
# 8. Sweep, intervals, baseline and checks
"""Regenerate every results figure from code, for AttackAware PolyIoM v1.1.4.

Loaded by AttackAware_PolyIoM_v1_1_4_ALL_FIGURES.ipynb after the study's
own runtime (ablation_only.py, scoredump_only.py, inversion_only.py and
revocation_only.py). Those files compute the ablation, the score
histograms, the inversion attack and the revocation test exactly as the
study did. This file adds the three computations they do not cover, and
the checks:

  sweep        the 80-setting development sweep behind Figures 3, 4, S2
  intervals    the identity-cluster bootstrap intervals behind Figure 5
  baseline     protected against unprotected EER, behind Figure P

Inputs are read from RUNS_IN (the study's runs/ folder) and every output
is written to DIR["runs"], which the notebook points at a separate folder.
Nothing in runs/ or seal/ is ever written.

After each stage the regenerated result is compared with the study's
original output, where that output is on Drive, and the differences are
printed. A figure is therefore only drawn from numbers this notebook has
computed and then checked.
"""

# ------------------------------------------------------------ input paths
# The study's runtime reads its inputs through DIR["runs"]. The notebook
# redirects DIR["runs"] to a fresh output folder, so the two readers below
# are redefined to keep reading the sealed inputs from RUNS_IN.

def selected_poly_key(modality):
    path = RUNS_IN / "key_search" / modality / "selected_key.json"
    obj = json.loads(path.read_text())
    return np.asarray(obj["C"], np.float32), np.asarray(obj["E"], np.int64)


def load_iom_tensor(modality, overlap, d):
    k = 1 + math.ceil((d - G) / (G - overlap))
    path = RUNS_IN / "iom_tensors" / f"{modality}_o{overlap}_k{k}.pt"
    if not path.is_file():
        raise FileNotFoundError(f"Frozen IoM tensor is missing: {path}")
    try:
        tensor = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        tensor = torch.load(path, map_location="cpu")
    if (tensor.ndim != 3 or tensor.shape[0] < 256 or tensor.shape[1] < 32
            or tensor.shape[-1] != k):
        raise RuntimeError(f"Frozen IoM tensor has an invalid shape: {path}")
    return tensor


VCTK_EMB = DIR["embeddings"] / "vctk_external_embeddings.npz"


def vctk_external_data():
    """External VCTK identities, built as the sealed external evaluation
    built them: ranks 0-4 averaged for enrolment, ranks 5-14 as probes."""
    z = np.load(VCTK_EMB, allow_pickle=False)
    E = z["embedding"].astype(np.float32)
    table = pd.DataFrame({
        "speaker_id": z["speaker_id"].astype(str),
        "rank": z["rank"].astype(int),
        "embedding_index": z["embedding_index"].astype(int),
    })
    subjects, enroll, probes = [], {}, {}
    for uid, g in table.groupby("speaker_id"):
        g = g.sort_values("rank")
        if len(g) != 15:
            continue
        uid = str(uid)
        members = np.stack([E[int(r.embedding_index)]
                            for r in g[g["rank"] < 5].itertuples(index=False)])
        subjects.append(uid)
        enroll[uid] = l2_np(np.mean(members, axis=0))
        probes[uid] = [E[int(r.embedding_index)]
                       for r in g[g["rank"] >= 5].itertuples(index=False)]
    return sorted(subjects), enroll, probes


# ------------------------------------------- weighted metrics (unchanged)
# From external_confidence_cells.py, the study's identity_cluster_percentile_v1
# method. A genuine trial of identity i carries weight w_i; an ordered
# impostor pair (a, b) carries w_a * w_b; same-identity pairs never form.

def _tar_w(gen_hist, tau):
    total = gen_hist.sum()
    return float(gen_hist[tau:].sum() / total) if total > 0 else float("nan")


def _fmr_w(imp_hist, tau):
    total = imp_hist.sum()
    return float(imp_hist[tau:].sum() / total) if total > 0 else float("nan")


def _eer_w(gen_hist, imp_hist, M):
    g_total, i_total = gen_hist.sum(), imp_hist.sum()
    if g_total <= 0 or i_total <= 0:
        return float("nan"), -1
    imp_tail = np.concatenate([imp_hist[::-1].cumsum()[::-1], [0.0]])
    gen_head = np.concatenate([[0.0], gen_hist.cumsum()])
    fmr = imp_tail / i_total
    fnmr = gen_head / g_total
    diff = fmr - fnmr
    cross = int(np.argmax(diff <= 0))
    if diff[cross] > 0:
        return float(fmr[-1]), M
    if cross == 0:
        return float((fmr[0] + fnmr[0]) / 2.0), 0
    d1, d2 = diff[cross - 1], diff[cross]
    t = 0.0 if d1 == d2 else float(d1 / (d1 - d2))
    return float(fmr[cross - 1] + t * (fmr[cross] - fmr[cross - 1])), cross


def _dsys_w(mated_hist, nonmated_hist, M):
    m_total, n_total = mated_hist.sum(), nonmated_hist.sum()
    if m_total <= 0 or n_total <= 0:
        return float("nan")
    p_m = mated_hist / m_total
    p_n = nonmated_hist / n_total
    denom = p_m + p_n
    with np.errstate(divide="ignore", invalid="ignore"):
        d = np.where(denom > 0, 2.0 * p_m / denom - 1.0, 0.0)
    return float(np.sum(p_m * np.clip(d, 0.0, 1.0)))


def _apply_weights(H, w):
    g = w @ H["Ghist"]
    i = w @ np.tensordot(w, H["Ihist"], axes=(0, 0))
    m = w @ H["Mhist"]
    nm = w @ np.tensordot(w, H["Nhist"], axes=(0, 0))
    return g, i, m, nm


def _metrics(H, w):
    g, i, m, nm = _apply_weights(H, w)
    eer, c_eer = _eer_w(g, i, H["M"])
    return {"EER": eer, "c_EER": c_eer,
            "TAR": _tar_w(g, H["tau"]), "FMR": _fmr_w(i, H["tau"]),
            "Dsys": _dsys_w(m, nm, H["M"])}


# ------------------------------------- per-identity score histograms
def unlink_tensor(modality, index, M, q, k, variant="shape"):
    """One of the two independently seeded unlinkability keys.

    "shape" draws exactly (M, q, k), as the held-out evaluation and the
    ablation do. "slice" draws the largest grid (256, 32, k) and slices it;
    the sweep stage tries it only if "shape" fails to reproduce a row.
    """
    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed_unlink(modality, index))
    if variant == "shape":
        return torch.randn((M, q, k), generator=generator, dtype=torch.float32)
    full = torch.randn((max(GRID["M"]), max(GRID["q"]), k),
                       generator=generator, dtype=torch.float32)
    return full[:M, :q].contiguous()


def identity_histograms(modality, subjects, enroll, probes, M, q, o, tau,
                        frozen=None, variant="shape"):
    """Genuine, impostor, mated and non-mated score histograms per identity,
    from the sealed pipeline (polynomial K*, frozen IoM tensor R)."""
    key = selected_poly_key(modality)
    n, d = len(subjects), len(enroll[subjects[0]])
    k = 1 + math.ceil((d - G) / (G - o))
    if frozen is None:
        frozen = load_iom_tensor(modality, o, d)
    recognition = frozen[:M, :q].to(DEVICE, torch.float32)

    enrolment = [enroll[u] for u in subjects]
    probe_vectors, owners = [], []
    for index, u in enumerate(subjects):
        probe_vectors.extend(probes[u])
        owners.extend([index] * len(probes[u]))
    owners = np.asarray(owners)
    enrolled = hash_arm(enrolment, "polyiom", key, o, recognition, M, q, None)
    hashed = hash_arm(probe_vectors, "polyiom", key, o, recognition, M, q, None)
    collision = (hashed[:, None, :] == enrolled[None, :, :]).sum(axis=-1)

    bins = M + 1
    Ghist = np.zeros((n, bins))
    Ihist = np.zeros((n, n, bins))           # [claimed, true, score]
    rows = np.arange(len(owners))
    np.add.at(Ghist, (owners, collision[rows, owners]), 1.0)
    for claimed in range(n):
        other = owners != claimed
        np.add.at(Ihist, (claimed, owners[other], collision[other, claimed]), 1.0)

    first = hash_arm(enrolment, "polyiom", key, o,
                     unlink_tensor(modality, 1, M, q, k, variant).to(DEVICE),
                     M, q, None)
    second = hash_arm(enrolment, "polyiom", key, o,
                      unlink_tensor(modality, 2, M, q, k, variant).to(DEVICE),
                      M, q, None)
    cross = (first[:, None, :] == second[None, :, :]).sum(axis=-1)
    Mhist = np.zeros((n, bins))
    Nhist = np.zeros((n, n, bins))
    Mhist[np.arange(n), np.diag(cross)] = 1.0
    for a in range(n):
        for b in range(n):
            if a != b:
                Nhist[a, b, cross[a, b]] = 1.0
    return {"subjects": subjects, "n": n, "M": M, "q": q, "o": o, "tau": tau,
            "Ghist": Ghist, "Ihist": Ihist, "Mhist": Mhist, "Nhist": Nhist}


# ----------------------------------------------------------------- sweep
def _config():
    path = DIR["seal"] / "config_v1_1_4.json"
    if path.is_file():
        return json.loads(path.read_text())
    print("  seal/config_v1_1_4.json not found; using the documented grid")
    return {"M_grid": [32, 64, 128, 256], "q_grid": [4, 8, 16, 32],
            "overlaps": [0, 1, 2, 3, 4], "target_fmr": 0.001}


CONFIG = _config()
GRID = {"M": CONFIG["M_grid"], "q": CONFIG["q_grid"], "o": CONFIG["overlaps"]}
SWEEP_FIELDS = ("EER", "c_EER", "c_tau_DEV", "TAR_DEV", "FMR_DEV",
                "FNMR_DEV", "Dsys_DEV")


def dev_threshold(imp_hist, target):
    """The lowest score at which the development FMR meets the target."""
    total = imp_hist.sum()
    for c in range(len(imp_hist) + 1):
        if imp_hist[c:].sum() / total <= target + 1e-12:
            return c
    return len(imp_hist)


def sweep_row(modality, H):
    ones = np.ones(H["n"])
    g, i, m, nm = _apply_weights(H, ones)
    eer, c_eer = _eer_w(g, i, H["M"])
    tau = dev_threshold(i, CONFIG["target_fmr"])
    tar = _tar_w(g, tau)
    return {"modality": modality, "M": H["M"], "q": H["q"], "o": H["o"],
            "n_subjects": H["n"], "n_genuine": int(g.sum()),
            "n_impostor": int(i.sum()), "c_tau_DEV": tau, "EER": eer,
            "c_EER": c_eer, "TAR_DEV": tar, "FMR_DEV": _fmr_w(i, tau),
            "FNMR_DEV": 1.0 - tar, "Dsys_DEV": _dsys_w(m, nm, H["M"])}


def _row_matches(new, old, tol=1e-9):
    return all(abs(float(new[f]) - float(old[f])) <= tol for f in SWEEP_FIELDS)


def run_sweep(modality):
    """Recompute the 80-setting development sweep and compare it, row by
    row, with the sealed runs/sweep/<modality>/sweep_80.csv."""
    out = DIR["runs"] / "sweep" / modality / "sweep_80.csv"
    if out.is_file():
        print(f"  {modality}: resume - regenerated sweep found")
        rows = pd.read_csv(out).to_dict("records")
    else:
        subjects, enroll, probes = partition_data(modality, "development")
        d = len(enroll[subjects[0]])
        sealed_path = RUNS_IN / "sweep" / modality / "sweep_80.csv"
        sealed = ({(r["M"], r["q"], r["o"]): r for r in
                   pd.read_csv(sealed_path).to_dict("records")}
                  if sealed_path.is_file() else {})
        rows = []
        for o in GRID["o"]:
            frozen = load_iom_tensor(modality, o, d)
            for M in GRID["M"]:
                for q in GRID["q"]:
                    H = identity_histograms(modality, subjects, enroll, probes,
                                            M, q, o, None, frozen=frozen)
                    row = sweep_row(modality, H)
                    old = sealed.get((M, q, o))
                    if old is not None and not _row_matches(row, old):
                        H2 = identity_histograms(modality, subjects, enroll,
                                                 probes, M, q, o, None,
                                                 frozen=frozen, variant="slice")
                        row2 = sweep_row(modality, H2)
                        if _row_matches(row2, old):
                            row = row2
                    rows.append(row)
            print(f"    {modality} o={o}: {len(rows)} of "
                  f"{len(GRID['o']) * len(GRID['M']) * len(GRID['q'])} settings")
        out.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(rows).to_csv(out, index=False)
    compare_sweep(modality, rows)
    return rows


def compare_sweep(modality, rows):
    path = RUNS_IN / "sweep" / modality / "sweep_80.csv"
    if not path.is_file():
        print(f"  {modality}: no sealed sweep on Drive to compare against")
        return None
    sealed = {(r["M"], r["q"], r["o"]): r
              for r in pd.read_csv(path).to_dict("records")}
    same = [r for r in rows if (r["M"], r["q"], r["o"]) in sealed
            and _row_matches(r, sealed[(r["M"], r["q"], r["o"])])]
    print(f"  {modality}: {len(same)} of {len(rows)} regenerated settings equal "
          f"the sealed sweep on {', '.join(SWEEP_FIELDS)}")
    for r in rows:
        old = sealed.get((r["M"], r["q"], r["o"]))
        if old is not None and not _row_matches(r, old):
            diffs = {f: (round(float(r[f]), 6), round(float(old[f]), 6))
                     for f in SWEEP_FIELDS
                     if abs(float(r[f]) - float(old[f])) > 1e-9}
            print(f"    differs M={r['M']} q={r['q']} o={r['o']}: {diffs}")
    return len(same) == len(rows)


def floor_of(modality):
    path = DIR["seal"] / f"operating_point_{modality}.json"
    if path.is_file():
        rule = json.loads(path.read_text())["rule"]
        return float(rule["eer_max"]), float(rule["tar_min"])
    raise FileNotFoundError(f"sealed operating point not found: {path}")


def select(modality, rows, floor):
    """The study's rule: lowest Dsys among settings meeting the floor; ties
    by lower EER, then smaller M, q, o."""
    eer_max, tar_min = floor
    ok = [r for r in rows if r["EER"] <= eer_max and r["TAR_DEV"] >= tar_min]
    if not ok:
        raise RuntimeError(f"{modality}: no setting meets the floor {floor}")
    best = min(ok, key=lambda r: (r["Dsys_DEV"], r["EER"], r["M"], r["q"], r["o"]))
    sealed = json.loads(HELDOUT_RESULTS[modality].read_text())
    chosen = (best["M"], best["q"], best["o"])
    expected = (sealed["M"], sealed["q"], sealed["o"])
    tau_ok = int(best["c_tau_DEV"]) == int(sealed["c_tau_carried_from_DEV"])
    print(f"  {modality}: {len(ok)} settings meet the floor; the rule selects "
          f"M={chosen[0]} q={chosen[1]} o={chosen[2]} with tau={best['c_tau_DEV']}"
          f" - {'as sealed' if chosen == expected and tau_ok else '*** NOT the sealed setting'}")
    if chosen != expected or not tau_ok:
        raise RuntimeError(f"{modality}: regenerated selection {chosen} "
                           f"differs from the sealed {expected}")
    return best


# ------------------------------------------------------------- intervals
# Seeds of the study's held-out interval run, as recorded in
# RESULTS_SUMMARY.md; used only if runs/heldout/heldout_confidence.json
# does not record them itself. The external seed is derived by the
# study's own rule.
HELDOUT_SEEDS = {"face": 6353163838588607505, "voice": 4722292304065993655}


def external_seed(modality):
    label = f"{S0}|external_confidence|{modality}".encode()
    return int.from_bytes(hashlib.sha256(label).digest()[:8], "big") >> 1


def _original_interval_file(partition):
    path = (RUNS_IN / "heldout" / "heldout_confidence.json"
            if partition == "evaluation"
            else RUNS_IN / "external" / "voice" / "external_confidence.json")
    return json.loads(path.read_text()) if path.is_file() else None


def _seed_for(modality, partition):
    original = _original_interval_file(partition)
    if original:
        recorded = original.get("modalities", {}).get(modality, {}) \
            .get("bootstrap_seed")
        if recorded is not None:
            return int(recorded)
    return (HELDOUT_SEEDS[modality] if partition == "evaluation"
            else external_seed(modality))


def interval_for(modality, partition, n_bootstrap, confidence):
    if partition == "evaluation":
        sealed = json.loads(HELDOUT_RESULTS[modality].read_text())
        tag = "HOLDOUT"
        subjects, enroll, probes = (face_heldout_data() if modality == "face"
                                    else voice_heldout_data())
    else:
        sealed = json.loads((RUNS_IN / "external" / modality /
                             "external_result.json").read_text())
        tag = "EXT"
        subjects, enroll, probes = vctk_external_data()
    M, q, o = sealed["M"], sealed["q"], sealed["o"]
    tau = int(sealed["c_tau_carried_from_DEV"])
    H = identity_histograms(modality, subjects, enroll, probes, M, q, o, tau)

    point = _metrics(H, np.ones(H["n"]))
    sealed_point = {"EER": sealed[f"EER_{tag}"],
                    "TAR": sealed[f"TAR_{tag}_at_dev_threshold"],
                    "FMR": sealed[f"FMR_{tag}_at_dev_threshold"],
                    "Dsys": sealed[f"Dsys_{tag}"]}
    drift = {k: (point[k], v) for k, v in sealed_point.items()
             if abs(point[k] - v) > 1e-6}
    if drift:
        raise RuntimeError(f"{modality}/{partition}: rebuilt scores do not "
                           f"reproduce the sealed result: {drift}")

    seed = _seed_for(modality, partition)
    rng = np.random.default_rng(seed)
    keys = ("EER", "TAR", "FMR", "Dsys")
    draws = {k: np.empty(n_bootstrap) for k in keys}
    for b in range(n_bootstrap):
        w = rng.multinomial(H["n"], np.full(H["n"], 1.0 / H["n"])).astype(float)
        mb = _metrics(H, w)
        for k in keys:
            draws[k][b] = mb[k]
    alpha = (1.0 - confidence) / 2.0
    out = {}
    for k in keys:
        d = draws[k][np.isfinite(draws[k])]
        out[k] = {"estimate": float(point[k]),
                  "lower": float(np.percentile(d, 100 * alpha)),
                  "upper": float(np.percentile(d, 100 * (1 - alpha)))}
    print(f"  {modality}/{partition}: {H['n']} identities, seed {seed}; "
          f"point estimates reproduce the sealed result")
    return {"metrics": out, "bootstrap_seed": seed, "n_subjects": H["n"]}


def run_intervals(n_bootstrap=2000, confidence=0.95):
    path = DIR["runs"] / "intervals" / "confidence_intervals.json"
    if path.is_file():
        print("Resume: regenerated intervals found; returning them unchanged.")
        return json.loads(path.read_text())
    result = {"schema": 1, "method_id": "identity_cluster_percentile_v1",
              "n_bootstrap": n_bootstrap, "confidence": confidence,
              "partitions": {}}
    parts = [("voice", "evaluation"), ("face", "evaluation")]
    if VCTK_EMB.is_file() and (RUNS_IN / "external" / "voice" /
                               "external_result.json").is_file():
        parts.append(("voice", "external"))
    else:
        print("  voice/external: VCTK embeddings or sealed external result "
              "not found; skipped")
    for modality, partition in parts:
        result["partitions"][f"{modality}/{partition}"] = interval_for(
            modality, partition, n_bootstrap, confidence)
    atomic_json(path, result)
    compare_intervals(result)
    return result


def compare_intervals(result):
    names = {"EER": "EER_{}", "TAR": "TAR_{}_at_dev_threshold",
             "FMR": "FMR_{}_at_dev_threshold", "Dsys": "Dsys_{}"}
    for part, rec in result["partitions"].items():
        modality, partition = part.split("/")
        original = _original_interval_file(partition)
        if not original:
            print(f"  {part}: no original interval file on Drive to compare")
            continue
        tag = "HOLDOUT" if partition == "evaluation" else "EXT"
        old = original["modalities"][modality]["metrics"]
        worst = max(abs(rec["metrics"][k][side] - old[names[k].format(tag)][side])
                    for k in names for side in ("estimate", "lower", "upper"))
        print(f"  {part}: largest difference from the original intervals "
              f"{worst:.2e}" + ("  (identical)" if worst <= 1e-12 else ""))


def intervals_for_plots(result):
    """Percent for EER, TAR, FMR; fraction for Dsys; as (point, low, high)."""
    out = {}
    for part, rec in result["partitions"].items():
        m = rec["metrics"]
        out[part] = {k: tuple(m[k][s] * (1 if k == "Dsys" else 100)
                              for s in ("estimate", "lower", "upper"))
                     for k in ("EER", "TAR", "FMR", "Dsys")}
    return out


# -------------------------------------------------------------- baseline
def clopper_pearson_upper(k, n, confidence=0.95):
    from scipy.stats import beta
    return float(beta.ppf(confidence, k + 1, n - k))


def baseline_from_scores(scores):
    """Protected against unprotected EER, from the score histograms.

    Where the unprotected system makes fewer than one genuine error, its
    EER is not resolvable and only a one-sided 95% upper bound is given.
    """
    rows = []
    for modality in ("voice", "face"):
        entry = scores["modalities"][modality]
        for partition in ("development", "evaluation", "external"):
            h = entry["partitions"].get(partition)
            if h is None:
                continue
            protected = eer_check(h["genuine"], h["impostor"], entry["M"])
            raw = h["unprotected"]["EER"]
            n = h["n_genuine"]
            k = int(math.floor(raw * n + 1e-9))
            rows.append({
                "modality": modality, "partition": partition,
                "n_genuine": n,
                "protected_eer": protected * 100,
                "unprotected_eer": raw * 100,
                "cost_pp": (protected - raw) * 100,
                "one_genuine_error_pct": 100.0 / n,
                "baseline_resolved": bool(raw >= 1.0 / n),
                "unprotected_upper_bound_95": clopper_pearson_upper(k, n) * 100,
            })
    return rows


def add_external_scores(scores):
    """The study's score dump adds VCTK only through a helper file kept on
    Drive. If it did not, build the external partition here, the same way."""
    voice = scores["modalities"]["voice"]
    if "external" in voice["partitions"] or not VCTK_EMB.is_file():
        return scores
    subjects, enroll, probes = vctk_external_data()
    M, q, o = voice["M"], voice["q"], voice["o"]
    d = len(enroll[subjects[0]])
    k = 1 + math.ceil((d - G) / (G - o))
    recognition = load_iom_tensor("voice", o, d)[:M, :q].to(DEVICE, torch.float32)
    pair = [unlink_tensor("voice", i, M, q, k).to(DEVICE) for i in (1, 2)]
    h = histograms_for("voice", subjects, enroll, probes, M, q, o,
                       selected_poly_key("voice"), recognition, pair)
    h["dlink_curve"], h["dlink_global"] = unlinkability_curve(h["mated"],
                                                              h["nonmated"])
    h["unprotected"] = cosine_histograms(subjects, enroll, probes)
    h["unprotected"]["EER"] = eer_cosine(h["unprotected"]["genuine"],
                                         h["unprotected"]["impostor"])
    voice["partitions"]["external"] = h
    atomic_json(DIR["runs"] / "scores" / "score_histograms.json", scores)
    print(f"  external: {h['n_subjects']} speakers added to the score dump")
    return scores


# ------------------------------------------------------------ comparison
def _leaves(obj, prefix=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from _leaves(v, f"{prefix}/{k}")
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from _leaves(v, f"{prefix}[{i}]")
    else:
        yield prefix, obj


def compare_with_original(name, regenerated, relative):
    """Every number in a regenerated result against the study's original."""
    path = RUNS_IN / relative
    if not path.is_file():
        print(f"  {name:<22} no original on Drive to compare against")
        return None
    old = dict(_leaves(json.loads(path.read_text())))
    new = dict(_leaves(regenerated))
    numeric = [k for k in old if isinstance(old[k], (int, float))
               and not isinstance(old[k], bool) and k in new]
    worst, where = 0.0, None
    for k in numeric:
        diff = abs(float(new[k]) - float(old[k]))
        if diff > worst:
            worst, where = diff, k
    missing = [k for k in old if k not in new]
    verdict = "identical" if worst == 0.0 and not missing else (
        f"largest difference {worst:.2e} at {where}")
    print(f"  {name:<22} {len(numeric):6d} numbers compared: {verdict}"
          + (f"; {len(missing)} fields absent" if missing else ""))
    return worst


# ---------------------------------------------------------------- figures
def draw_figures(out, sweeps, selected, floors, intervals, baseline,
                 ablation, inversion, revocation, score_json):
    data = {"sweep": sweeps, "floor": floors, "selected": selected,
            "intervals": intervals_for_plots(intervals), "baseline": baseline,
            "ablation": ablation["modalities"], "revocation": revocation,
            "inversion": inversion["modalities"]}
    stems = results_plots.draw_all(data, out)

    # Figures A-D: the study's own figure code, given the intervals this
    # notebook computed instead of the ones typed into it.
    figures_only.CARRIED_CI = {
        tuple(part.split("/")): data["intervals"][part]["EER"]
        for part in data["intervals"]}
    figures_only.make_figures(score_json, out)
    return stems + ["figA_det_preservation", "figB_score_separability",
                    "figC_unlinkability", "figD_revocability"]


print("All-figures runtime ready.")


In [ ]:
#@title 9. Load the runtime
for name in ("ablation_only.py", "scoredump_only.py", "inversion_only.py",
             "revocation_only.py", "allfigures_only.py"):
    exec(compile((RUNTIME / name).read_text(), str(RUNTIME / name), "exec"),
         globals())


def _module(name):
    spec = importlib.util.spec_from_file_location(name, RUNTIME / f"{name}.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


# Imported as modules so their names cannot collide with the study's.
results_plots = _module("results_plots")
figures_only = _module("figures_only")
print("Runtime loaded.")


In [ ]:
#@title 10. Stage 1 - the development sweep (Figures 3, 4, S2)
SWEEPS, SELECTED, FLOORS = {}, {}, {}
for modality in ("voice", "face"):
    SWEEPS[modality] = run_sweep(modality)
    FLOORS[modality] = floor_of(modality)
    SELECTED[modality] = select(modality, SWEEPS[modality], FLOORS[modality])


In [ ]:
#@title 11. Stage 2 - score histograms, protected and unprotected (Figures A-D)
SCORES = add_external_scores(run_score_dump())


In [ ]:
#@title 12. Stage 3 - intervals and protection cost (Figures 5, P)
INTERVALS = run_intervals(N_BOOTSTRAP, CONFIDENCE)
BASELINE = baseline_from_scores(SCORES)
display(pd.DataFrame(BASELINE).round(4))


In [ ]:
#@title 13. Stage 4 - ablation (Figure 6)
ABLATION = run_ablation(N_BOOTSTRAP, CONFIDENCE)
contrast_report(ABLATION)


In [ ]:
#@title 14. Stage 5 - full-knowledge inversion attack (Figure 8)
INVERSION = run_inversion(N_BOOTSTRAP, CONFIDENCE)
inversion_report(INVERSION)


In [ ]:
#@title 15. Stage 6 - revocation over fresh keys (Figure 7)
REVOCATION = run_revocation(N_BOOTSTRAP, CONFIDENCE)
revocation_report(REVOCATION)


In [ ]:
#@title 16. Stage 7 - compare with the study's original outputs
compare_with_original("score histograms", SCORES, "scores/score_histograms.json")
compare_with_original("ablation", ABLATION, "ablation/ablation_result.json")
compare_with_original("inversion", INVERSION, "inversion/inversion_result.json")
compare_with_original("revocation", REVOCATION, "revocation/revocation_result.json")


In [ ]:
#@title 17. Stage 8 - draw every figure, then download them
STEMS = draw_figures(FIGURES_OUT, SWEEPS, SELECTED, FLOORS, INTERVALS,
                     BASELINE, ABLATION, INVERSION, REVOCATION,
                     WORK / "scores" / "score_histograms.json")
archive = shutil.make_archive(str(PROJECT / "AttackAware_PolyIoM_figures_regenerated"),
                              "zip", FIGURES_OUT)
print(f"\n{len(STEMS)} figures, PNG and PDF, in {FIGURES_OUT}")
print(f"zip: {archive}")
if IN_COLAB:
    files.download(archive)
